# 01 — Generación de variables con IA

Acá se convierten las conversaciones anonimizadas en variables estructuradas
usando un LLM bajo un contrato que se puede validar (`esquema.py` +
`prompts.py`), y se arman las tablas que responden las cinco preguntas del
enunciado.

| Pregunta | Variables que la responden |
|---|---|
| **(a)** Principales motivos de no pago | `motivo_no_pago`, `motivos_secundarios`, `evidencia_motivo` |
| **(b)** Ofertas que hacen los asesores | `ofertas_asesor` → `ofertas_largo.csv` |
| **(c)** Argumentos que logran más acuerdos | `ofertas_asesor` × `acuerdo_pago`, `argumentos_asesor` |
| **(d)** Qué pasa en cada conversación | `resumen` |
| **(e)** Las peor calificadas y por qué | `score_satisfaccion`, `factores_insatisfaccion`, `recomendacion_mejora` |

## Metodología y decisiones de diseño

## El enfoque: extracción estructurada con contrato validable

En vez de pedirle al modelo texto libre y limpiarlo después, se define primero
un **contrato de salida** (`esquema.py`) y se lo obliga a cumplirlo. Las
cuatro taxonomías son cerradas (`Enum`), no listas abiertas, por una razón muy
concreta: las preguntas (a), (b) y (c) son rankings de frecuencia, y con
etiquetas libres el mismo motivo termina escrito de veinte formas distintas
hasta que el conteo deja de servir para algo.

Las categorías no se inventaron: salen de frecuencias medidas sobre el corpus
en el EDA (`esquema.TRAZABILIDAD_TAXONOMIAS` deja la trazabilidad).

## Decisiones de diseño

**La conversación completa es la unidad de análisis**, no el mensaje suelto.
Lo que se busca casi siempre depende del hilo — el motivo que da el cliente en
el mensaje 4 explica la oferta que aparece en el 30 — así que cada
conversación se procesa entera en una sola llamada.

**El orden de los mensajes sale de `uk_id_mensaje`.** No hay timestamp en la
base, así que el orden se reconstruye con el consecutivo numérico de ese
campo, el mismo criterio ya validado en el EDA.

**El acuerdo de pago se valida con reglas, no con criterio del modelo.** Solo
cuenta cuando el cliente manifiesta explícitamente un compromiso e indica una
fecha específica — es la definición literal del enunciado, y se impone tanto
en el prompt como en validadores de Pydantic que rechazan cualquier respuesta
incoherente.

**Toda clasificación interpretativa necesita una cita.** Si el modelo no
puede señalar la frase que respalda una categoría, la categoría es
`NINGUNO`. Preferimos un dato ausente a una atribución inventada.

**La satisfacción combina el juicio del modelo con señales objetivas** —70%
LLM, 30% métricas medidas sobre los mensajes— para corregir el sesgo conocido
de los LLM como jueces: comprimen sus notas hacia el centro de la escala. La
sección 7 explica el cálculo completo.

**La corrida es reproducible y reanudable.** `temperature=0` y una semilla
fija bajan la variabilidad entre corridas, y cada resultado se guarda en
JSONL apenas se obtiene, así que interrumpir el proceso no obliga a repetir
lo ya hecho ni a pagarlo de nuevo.

**La calidad se mide, no se asume.** Hay una muestra anotada a mano, una
prueba de estabilidad entre corridas y una comparación contra el NPS que el
propio bot recoge — tres formas independientes de poner a prueba lo que sale
del modelo.

## Modelo y proveedor: la historia real de esta corrida

El plan original era Gemini AI Studio, porque su free tier alcanzaba para una
muestra estratificada del corpus. Antes de llegar ahí se probaron tres
proveedores gratuitos que resultaron inviables: Groq daba 429 permanente
porque su límite de tokens por minuto es menor que el prompt fijo, el free
tier de Mistral quedó descontinuado a mitad de esta prueba técnica, y
Cerebras devolvía `payment_required` desde la primera llamada (bitácora v11
en `prompts.py`). Con Gemini sí se corrió un piloto de 15 conversaciones —
funcionó bien— pero su tope diario de peticiones obligaba a resignarse a una
muestra en vez del corpus completo.

Cuando hubo presupuesto real, el plan cambió: se migró a la API de pago de
OpenAI y se procesó el corpus completo. La primera corrida usó
`gpt-4.1-nano`, el modelo más barato del catálogo, razonable para una tarea
de extracción guiada por un contrato cerrado. Terminó sin errores, pero una
auditoría posterior encontró dos problemas serios: el modelo sub-extraía
`motivo_no_pago` —devolvía `NINGUNO` en el 93% de los casos, incluso cuando el
cliente decía "ya pagué" con esas palabras— y un hueco en la validación de
`acuerdo_pago` dejaba pasar como acuerdo del cliente frases que en realidad
decía el asesor ("te confirmo el compromiso de pago para el..."). Ninguno de
los dos hallazgos lo produjo el modelo por sí solo: el segundo es un defecto
del contrato (`esquema.py` no comprobaba quién decía la frase), y se corrigió
sin gastar una sola llamada más. El primero sí exigía un modelo más capaz, así
que se repitió la corrida completa con `gpt-4o-mini` (bitácora v13). El
`motivo_no_pago` bajó a 63% de `NINGUNO` y la distribución por fin dice algo:
ingresos insuficientes, desacuerdo con el monto, descuento de nómina no
aplicado, desempleo.

El proveedor sigue siendo conmutable en una línea (`PROVEEDOR` en la celda del
cliente), y el resto del motor —reintentos, validación, caché— no cambia
según cuál se use.

## Limitaciones

- El corpus solo contiene conversaciones con 20 o más mensajes; los
  resultados no representan interacciones más cortas.
- No hay análisis temporal posible: `anio` y `mes` son constantes.
- Los datos están anonimizados con marcadores, y el proceso de anonimización
  tiene al menos un error conocido (rompe palabras del español) que se trató
  durante el preprocesamiento — ver la sección siguiente.

In [2]:
import json
import os
import re
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# esquema.py y prompts.py viven en la raiz del proyecto, un nivel arriba
RAIZ = Path("..").resolve()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import esquema
import prompts
from esquema import ConversacionIA

RUTA_EXCEL = RAIZ / "Prueba 2" / "Base sintetica conversaciones.xlsx"
DIR_OUT = RAIZ / "outputs"
DIR_OUT.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

print("esquema.py  :", len(esquema.CAMPOS), "campos en el contrato")
print("prompts.py  :", len(prompts.FEW_SHOT), "ejemplos few-shot validados contra Pydantic")
print("Excel       :", RUTA_EXCEL.name, "-", "encontrado" if RUTA_EXCEL.exists() else "NO ENCONTRADO")

esquema.py  : 22 campos en el contrato
prompts.py  : 3 ejemplos few-shot validados contra Pydantic
Excel       : Base sintetica conversaciones.xlsx - encontrado


---
## 1. Armado de las transcripciones

Se parte del Excel original y se aplican **las mismas decisiones de limpieza establecidas y justificadas en el EDA** (`00_EDA.ipynb`, secciones 2 y 4.1), para que ambos notebooks trabajen sobre exactamente el mismo corpus:

| Decisión | Motivo (verificado en el EDA) |
|---|---|
| Eliminar filas duplicadas exactas | Resuelve de paso los `uk_id_mensaje` repetidos |
| Descartar `uk_id_conversacion1` | Copia exacta de la llave |
| `mensaje` nulo → cadena vacía | Un solo caso |
| Reemplazar `\N` por espacio | Salto de línea escapado en el origen |
| Ordenar por el consecutivo de `uk_id_mensaje` | **No hay `timestamp`**; el consecutivo resultó monótono dentro de cada conversación |

El formato de salida es una línea por mensaje, `[ROL] texto`. La etiqueta de rol explícita es lo que le permite al modelo aplicar la regla más importante del prompt: **sólo el `AGENTE` genera ofertas y argumentos**; el `BOT` emite plantillas aunque se presente a sí mismo como asesor.

In [3]:
df = pd.read_excel(RUTA_EXCEL)
print(f"Filas crudas: {len(df):,}   Conversaciones: {df.uk_id_conversacion.nunique():,}")

#Limpieza, identica a la del EDA 
msg = (df.mensaje.fillna("")
                 .astype(str)
                 .str.replace(r"\\N", " ", regex=True)
                 .str.strip())
limpio = df.assign(msg=msg)
limpio = limpio.drop_duplicates(
    subset=[c for c in df.columns]           # duplicado exacto sobre columnas originales
).drop(columns=["uk_id_conversacion1"])

# Sin timestamp: el consecutivo de uk_id_mensaje define el orden
limpio["orden"] = limpio.uk_id_mensaje.str.extract(r"(\d+)").astype(int)
limpio = limpio.sort_values(["uk_id_conversacion", "orden"]).reset_index(drop=True)

monotono = limpio.groupby("uk_id_conversacion").orden.apply(lambda s: s.is_monotonic_increasing).all()
print(f"Filas tras limpieza: {len(limpio):,}  (se eliminaron {len(df)-len(limpio):,})")
print(f"Orden monotono dentro de cada conversacion: {monotono}")
print(f"Roles: {dict(limpio.de.value_counts())}")

Filas crudas: 42,607   Conversaciones: 1,197
Filas tras limpieza: 42,005  (se eliminaron 602)
Orden monotono dentro de cada conversacion: True
Roles: {'USUARIO': np.int64(18619), 'AGENTE': np.int64(13770), 'BOT': np.int64(8408), 'HSM': np.int64(1208)}


In [4]:
# Presupuesto de caracteres por transcripcion.
# En el EDA el percentil 95 de caracteres por conversacion resulto ~6.400, asi que
# 12.000 deja practicamente todo el corpus intacto. El truncado se hace POR EL MEDIO,
# no por el final: la apertura fija el motivo de contacto y el cierre concentra el
# compromiso de pago y el sentimiento final, que son justamente las variables criticas.
MAX_CARACTERES = 12_000


def armar_transcripcion(sub: pd.DataFrame, max_caracteres: int = MAX_CARACTERES):
    """Convierte los mensajes de una conversacion en texto '[ROL] mensaje' por linea.

    Devuelve (texto, fue_truncada).
    """
    lineas = [f"[{r.de}] {r.msg}" for r in sub.itertuples() if str(r.msg).strip()]
    texto = "\n".join(lineas)
    if len(texto) <= max_caracteres:
        return texto, False

    # Truncado por el medio: 60% del presupuesto al inicio, 40% al cierre.
    n_ini = int(max_caracteres * 0.60)
    n_fin = max_caracteres - n_ini
    acum, inicio = 0, []
    for ln in lineas:
        if acum + len(ln) > n_ini:
            break
        inicio.append(ln); acum += len(ln) + 1
    acum, final = 0, []
    for ln in reversed(lineas):
        if acum + len(ln) > n_fin:
            break
        final.insert(0, ln); acum += len(ln) + 1
    omitidos = len(lineas) - len(inicio) - len(final)
    marca = f"\n[... {omitidos} mensajes intermedios omitidos por longitud ...]\n"
    return "\n".join(inicio) + marca + "\n".join(final), True


transcripciones = {}
truncadas = []
for cid, sub in limpio.groupby("uk_id_conversacion", sort=False):
    texto, trunc = armar_transcripcion(sub)
    transcripciones[cid] = texto
    if trunc:
        truncadas.append(cid)

tr = pd.DataFrame({
    "uk_id_conversacion": list(transcripciones.keys()),
    "transcripcion": list(transcripciones.values()),
})
tr["n_caracteres"] = tr.transcripcion.str.len()
tr["n_lineas"] = tr.transcripcion.str.count("\n") + 1
tr["truncada"] = tr.uk_id_conversacion.isin(truncadas)

print(f"Transcripciones armadas: {len(tr):,}")
print(f"Truncadas por longitud:  {tr.truncada.sum()} ({tr.truncada.mean()*100:.2f}%)")
print()
print(tr[["n_caracteres", "n_lineas"]].describe().round(0).to_string())

Transcripciones armadas: 1,197
Truncadas por longitud:  5 (0.42%)

       n_caracteres  n_lineas
count        1197.0    1197.0
mean         3601.0      39.0
std          1688.0      21.0
min          1699.0      20.0
25%          2467.0      24.0
50%          3016.0      31.0
75%          4085.0      43.0
max         11946.0     177.0


In [5]:
# Inspeccion visual: una transcripcion tal como la recibira el modelo
ejemplo = tr.sort_values("n_caracteres").iloc[len(tr)//2]
print(f"uk_id_conversacion: {ejemplo.uk_id_conversacion}  |  {ejemplo.n_lineas} lineas  |  {ejemplo.n_caracteres} caracteres")
print("=" * 100)
print(ejemplo.transcripcion[:1800])
print("...")

uk_id_conversacion: CONV_00000075  |  32 lineas  |  3016 caracteres
[HSM] BANCO_000001 :BLUE_HEART: ¡HOLA!  TU OBLIGACIÓN PRESENTA CUOTAS PENDIENTES DE PAGO QUE REQUIEREN TU GESTIÓN. UNO DE NUESTROS ASESORES TIENE ALTERNATIVAS PARA QUE LOGRES NORMALIZAR TUS PAGOS ACORDE A TU CAPACIDAD DE PAGO  PERMÍTENOS AYUDARTE A RESOLVERLO OPORTUNAMENTE.'
[USUARIO] QUIERO VALIDAR MI CASO
[BOT] AL CONTINUAR NOS AUTORIZAS A QUE TUS DATOS SEAN TRATADOS CONFORME A LA LEY 1581 DEL FECHA_00000213 DE PROTECCIÓN DE DATOS QUE SE ENCUENTRA EN EL SIGUIENTE ENLACE: URL_ANONIMIZADA
[BOT] ESTAMOS VALIDANDO TU INFORMACIÓN. EN UN MOMENTO, UNO DE NUESTROS ASESORES TE AYUDARÁ A ENCONTRAR LA MEJOR ALTERNATIVA PARA QUE ESTÉS AL DÍA CON TUS OBLIGACIONES.
[BOT] NUESTRO ASESOR SE ENCUENTRA OCUPADO, ¿DESEAS SEGUIR ESPERANDO?
[BOT] HOLA, MI NOMBRE ES PERSONA_00000115 FORERO, SOY ASESOR DEL BANCO_000001. POR FAVOR INDÍCAME LA FECHA DE EXPEDICIÓN DE TÚ DOCUMENTO Y EL CORREO ELECTRÓNICO REGISTRADO.
[USUARIO] HOLA
[USUARIO] SI


### El anonimizador rompió palabras corrientes del español

Al perfilar los marcadores sobre el corpus aparece un problema de calidad que no estaba documentado y que, de no manejarse, sesga sistemáticamente la variable más importante del análisis.

**La anonimización reemplazó nombres propios por subcadena, sin exigir que fueran palabras completas.** Al hacerlo destruyó palabras comunes que contenían un nombre dentro. El daño alcanza **1.625 mensajes en 856 conversaciones — el 71,5 % del corpus**.

La celda siguiente lo verifica. Tres marcadores quedan decodificados con certeza:

| Marcador | Es realmente | Cómo se demuestra |
|---|---|---|
| `PERSONA_00000008` | **ANA** | Aparece pegado a fragmentos: `MAÑPERSONA_00000008` = MAÑANA (326 veces), `SEMPERSONA_00000008` = SEMANA (123), `DIPERSONA_00000008` = DIANA (163), más JULIANA, CERCANA, LEJANA, HERMANA, TATIANA, VIVIANA |
| `PERSONA_00000031` | **CESAR** | Queda *incrustado en mitad* de la palabra: `NEPERSONA_00000031IO` = NECE**SARIO**, `NEPERSONA_00000031IA` = NECESARIA (147 veces) |
| `PERSONA_00000108` | **JULIO** (un mes) | Al contar los meses literales del corpus están los doce menos uno: JUNIO 365, AGOSTO 361, MAYO 179… y **JULIO no aparece ni una vez**, pese a estar flanqueado por los dos meses más frecuentes. Se confirma en contexto: *"TRASLADO DE 2 CUOTAS, DE JUNIO Y PERSONA_00000108"* |

**Por qué esto no es una curiosidad sino un riesgo directo para el análisis.** Los tres marcadores caen justo sobre expresiones de tiempo, que son las que deciden `tipo_fecha` y por tanto `acuerdo_pago`:

| Lo que ve el modelo | Lo que realmente dice el cliente | Clasificación correcta |
|---|---|---|
| `MAÑPERSONA_00000008 HAGO EL PAGO` | *mañana hago el pago* | `FECHA_ESPECIFICA` → acuerdo |
| `LA PROXIMA SEMPERSONA_00000008 MIRO` | *la próxima semana miro* | `FECHA_VAGA` → no acuerdo |
| `PODRIA PAGARLO EL 25 DE PERSONA_00000108` | *el 25 de julio* | `FECHA_ESPECIFICA` → acuerdo |

Sin la regla, el modelo ve cadenas sin sentido, no reconoce la fecha y pierde el acuerdo. Como `acuerdo_pago` es el denominador de las preguntas (b) y (c), el error se propagaría a **todo** el análisis de efectividad de ofertas y argumentos.

Por eso `prompts.REGLA_ANONIMIZACION` incluye el glosario decodificado y la instrucción de reconstruir la palabra antes de interpretarla — aclarando que la reconstrucción es sólo para entender el sentido: las citas de evidencia se copian con el marcador intacto.

*Nota sobre el alcance de la corrección: se decodificaron los tres marcadores con evidencia concluyente. Es probable que haya otros nombres reemplazados por subcadena con menos apariciones; los tres identificados concentran los casos que afectan expresiones temporales, que son los que importan aquí.*

In [6]:
todo_el_texto = " ".join(limpio.msg)

#  1. Inventario de marcadores bien formados
marcadores = Counter(re.findall(r"\b([A-ZÁÉÍÓÚÑ]+)_(?:\d{6,}|ANONIMIZAD[AO])", todo_el_texto))
print("Marcadores de anonimizacion (tipos principales):")
for k, v in marcadores.most_common(10):
    print(f"  {k+'_':<14} {v:>6,}")

#  2. Magnitud del dano: marcadores pegados a fragmentos de palabra 
pegados = limpio.msg.str.contains(r"[A-ZÁÉÍÓÚÑ]+PERSONA_\d+", regex=True, na=False)
print(f"\nMensajes con un marcador PEGADO a letras: {pegados.sum():,} "
      f"en {limpio.loc[pegados, 'uk_id_conversacion'].nunique():,} conversaciones "
      f"({limpio.loc[pegados, 'uk_id_conversacion'].nunique()/limpio.uk_id_conversacion.nunique()*100:.1f}% del corpus)")

#  3. Decodificacion de PERSONA_00000008
print("\nPERSONA_00000008: prefijos con los que aparece pegado")
for pref, n in Counter(re.findall(r"([A-ZÁÉÍÓÚÑ]{1,12})PERSONA_00000008\b", todo_el_texto)).most_common(8):
    print(f"  {pref + 'PERSONA_00000008':<26} {n:>4}  ->  {pref}ANA")
print("  Conclusion: PERSONA_00000008 = 'ANA'")

# --- 4. Decodificacion de PERSONA_00000031 -------------------------------
print("\nPERSONA_00000031: aparece incrustado en mitad de palabra")
for ej in set(re.findall(r"[A-ZÁÉÍÓÚÑ]{0,6}PERSONA_00000031[A-ZÁÉÍÓÚÑ]{0,6}", todo_el_texto)):
    print(f"  {ej}")
print("  Conclusion: PERSONA_00000031 = 'CESAR'  (NE+CESAR+IO = NECESARIO)")

# --- 5. Decodificacion de PERSONA_00000108 -------------------------------
MESES = ["ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO", "JULIO",
         "AGOSTO", "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"]
conteo_meses = {m: len(re.findall(r"\b" + m + r"\b", todo_el_texto)) for m in MESES}
print("\nPERSONA_00000108: conteo de meses literales en el corpus")
print("  " + "  ".join(f"{m[:3]}={n}" for m, n in conteo_meses.items()))
ausentes = [m for m, n in conteo_meses.items() if n == 0]
print(f"  Meses que NO aparecen nunca: {ausentes}")
print(f"  '<dia> DE PERSONA_00000108': "
      f"{len(re.findall(r'\d{1,2} DE PERSONA_00000108', todo_el_texto))} apariciones")
print(f"  Conclusion: PERSONA_00000108 = '{ausentes[0] if len(ausentes) == 1 else '?'}'")
print("\n  Contextos que lo confirman:")
for mm in list(re.finditer(r".{45}PERSONA_00000108.{20}", todo_el_texto))[:4]:
    print(f"    ...{mm.group(0)}...")

print("\nGlosario aplicado en el prompt:", prompts.MARCADORES_DECODIFICADOS)

Marcadores de anonimizacion (tipos principales):
  BANCO_          4,049
  PERSONA_        3,493
  MONTO_          2,600
  FECHA_          2,453
  DOCUMENTO_      2,017
  URL_            1,799
  EMAIL_          1,278
  CREDITO_          915
  CPERSONA_         672
  MAÑPERSONA_       327

Mensajes con un marcador PEGADO a letras: 1,625 en 856 conversaciones (71.5% del corpus)

PERSONA_00000008: prefijos con los que aparece pegado
  MAÑPERSONA_00000008         326  ->  MAÑANA
  DIPERSONA_00000008          163  ->  DIANA
  SEMPERSONA_00000008         123  ->  SEMANA
  JULIPERSONA_00000008          9  ->  JULIANA
  CERCPERSONA_00000008          6  ->  CERCANA
  LEJPERSONA_00000008           6  ->  LEJANA
  ELIPERSONA_00000008           5  ->  ELIANA
  LILIPERSONA_00000008          4  ->  LILIANA
  Conclusion: PERSONA_00000008 = 'ANA'

PERSONA_00000031: aparece incrustado en mitad de palabra
  NEPERSONA_00000031IO
  NEPERSONA_00000031IAS
  NEPERSONA_00000031IOS
  INNEPERSONA_00000031IOS
  

In [7]:
# La tabla de transcripciones es un entregable en si mismo: alimenta el resumen de
# la pregunta (d) y permite auditar cualquier variable contra su texto de origen.
tr.to_csv(DIR_OUT / "transcripciones.csv", index=False, encoding="utf-8-sig")
print(f"Guardado: outputs/transcripciones.csv  ({len(tr):,} filas)")

Guardado: outputs/transcripciones.csv  (1,197 filas)


---
## 2. Estimación de costo y de tiempo — **antes** de lanzar la corrida

Esta celda existe para no descubrir a mitad de la corrida que el presupuesto o la cuota no alcanzan. Estima dos cosas distintas que se suelen confundir:

- **Costo**, gobernado por el total de tokens.
- **Tiempo**, gobernado por el límite de *tokens por minuto* (TPM) del plan, no por la latencia del modelo. La latencia por llamada es de segundos; el cuello de botella real es el TPM del plan y, en tiers gratuitos, el tope diario de tokens.

**Dos advertencias de honestidad sobre esta estimación:**

1. El conteo de tokens es **aproximado**. `tiktoken` no está instalado y, en todo caso, el tokenizador de gpt-oss no es exactamente el de OpenAI. Se usa una razón caracteres/token configurable. Para presupuestar está bien; no lo tome como una factura.
2. Los **precios y los límites de cuota son parámetros que usted debe llenar** con los valores reales de su cuenta (consola del proveedor, o los headers `x-ratelimit-*` de la primera respuesta). No se ponen valores por defecto inventados: la celda calcula el volumen de tokens —que sí es un dato derivado de los datos— y proyecta el costo para varios precios posibles.

In [8]:
# Limites del proveedor. Confirmar con los headers x-ratelimit-* de la 1a
# respuesta real, o en https://platform.openai.com/settings/organization/limits
# OpenAI API de pago, usage tier 1 (tras recargar >= $5). APROXIMADOS:
#   gpt-4.1-nano : el mas rapido y barato; suficiente para extraccion guiada
#   gpt-4o-mini / gpt-4.1-mini : mejor calidad, mas caros
RPM_LIMITE = 500
RPD_LIMITE = 100_000     # peticiones por dia  <- holgado: el corpus completo son ~1.700 llamadas
TPM_LIMITE = 200_000     # tokens por minuto   <- una llamada son ~11k: ~18 llamadas/min

RAZON_CHARS_POR_TOKEN = 3.6   # aproximacion para espanol en mayusculas
TOKENS_SALIDA_ESTIMADOS = 950 # el JSON de salida ronda este tamano (incluye analisis_previo)

# Volumen real, derivado de los datos
mensajes_fijos = prompts.construir_mensajes("", compacto=True, con_few_shot=True)
chars_prompt_fijo = sum(len(m["content"]) for m in mensajes_fijos)
tok_prompt_fijo = chars_prompt_fijo / RAZON_CHARS_POR_TOKEN

tok_transcripciones = tr.n_caracteres.sum() / RAZON_CHARS_POR_TOKEN
n_conv = len(tr)

tok_entrada_total = tok_prompt_fijo * n_conv + tok_transcripciones
tok_salida_total = TOKENS_SALIDA_ESTIMADOS * n_conv
tok_total = tok_entrada_total + tok_salida_total

print(f"Conversaciones en el corpus      : {n_conv:>12,}")
print(f"Prompt fijo (system + few-shot)  : {tok_prompt_fijo:>12,.0f} tokens/llamada")
print(f"Transcripcion media              : {tok_transcripciones/n_conv:>12,.0f} tokens/llamada")
print(f"  -> el prompt fijo es el {tok_prompt_fijo/(tok_prompt_fijo+tok_transcripciones/n_conv)*100:.0f}% de la entrada de cada llamada")
print()
print(f"Tokens TOTALES si se procesara TODO el corpus: {tok_total:>12,.0f}")

Conversaciones en el corpus      :        1,197
Prompt fijo (system + few-shot)  :        8,149 tokens/llamada
Transcripcion media              :        1,000 tokens/llamada
  -> el prompt fijo es el 89% de la entrada de cada llamada

Tokens TOTALES si se procesara TODO el corpus:   12,088,719


El prompt fijo domina la entrada. Es una consecuencia directa de no tener *prompt caching*: cada llamada vuelve a pagar las reglas de decisión y los dos ejemplos few-shot.

Esa concentración es también la palanca de ahorro más grande disponible. `prompts.construir_mensajes()` expone dos interruptores para negociar costo contra calidad:

| Configuración | Qué se pierde |
|---|---|
| `compacto=True` (por defecto) | Nada de la semántica: sólo se omite el JSON Schema verboso. Ahorro medido: ~1.952 tokens/llamada |
| `con_few_shot=False` | Sí se pierde calidad. Los ejemplos son los que enseñan las dos fronteras difíciles. **No recomendado sin medir antes el impacto en la muestra anotada** |

La celda siguiente proyecta costo y tiempo para las configuraciones disponibles.

In [9]:
from IPython.display import display

# --- Comparacion de configuraciones de prompt (volumen de tokens) ---------
def proyectar(nombre, tok_fijo_por_llamada):
    por_llamada = tok_fijo_por_llamada + tok_transcripciones / n_conv + TOKENS_SALIDA_ESTIMADOS
    ent = tok_fijo_por_llamada * n_conv + tok_transcripciones
    sal = TOKENS_SALIDA_ESTIMADOS * n_conv
    return {
        "configuracion": nombre,
        "tok_por_llamada": round(por_llamada),
        "cabe_en_TPM": "SI" if por_llamada <= TPM_LIMITE else "NO -- 429 permanente",
        "tokens_entrada": round(ent),
        "tokens_salida": round(sal),
        "tokens_total": round(ent + sal),
    }

msgs_sin_fs = prompts.construir_mensajes("", compacto=True, con_few_shot=False)
msgs_completo = prompts.construir_mensajes("", compacto=False, con_few_shot=True)
escenarios = pd.DataFrame([
    proyectar("compacto + few-shot  (produccion)", chars_prompt_fijo / RAZON_CHARS_POR_TOKEN),
    proyectar("compacto sin few-shot", sum(len(m["content"]) for m in msgs_sin_fs) / RAZON_CHARS_POR_TOKEN),
    proyectar("completo + few-shot", sum(len(m["content"]) for m in msgs_completo) / RAZON_CHARS_POR_TOKEN),
])
display(escenarios)

# --- Dimensionamiento de la MUESTRA (Gemini free tier) -------------------
# El limite que manda es RPD (peticiones/dia), no el TPM. Cada conversacion
# cuesta hasta 1 + MAX_REINTENTOS llamadas (reintento dirigido por validacion).
p = escenarios.iloc[0]
llamadas_por_conv = 1 + prompts.MAX_REINTENTOS
conv_por_dia = int(RPD_LIMITE / llamadas_por_conv)

print("DIMENSIONAMIENTO DE LA MUESTRA (Gemini free tier)")
print("=" * 70)
print(f"  Config de produccion: ~{p.tok_por_llamada:,} tok/llamada -> {p.cabe_en_TPM} (TPM {TPM_LIMITE:,})")
print(f"  RPD {RPD_LIMITE:,}  /  hasta {llamadas_por_conv} llamadas por conversacion  ->  ~{conv_por_dia} conversaciones/dia")
print()
for m in (100, 200, 300, 400, n_conv):
    et = "   <- corpus completo" if m == n_conv else ""
    print(f"  muestra de {m:>4}  ->  ~{m / conv_por_dia:.1f} dias{et}")
print()
print("  Recomendado: muestra ESTRATIFICADA de ~300 (por tramo de longitud y")
print("  presencia de AGENTE), procesable en ~1 dia. Los agregados se reportan")
print("  con su margen de error; para una prueba tecnica es defendible.")

,configuracion,tok_por_llamada,cabe_en_TPM,tokens_entrada,tokens_salida,tokens_total
0,compacto + few-shot (produccion),10099,SI,10951569,1137150,12088719
1,compacto sin few-shot,6778,SI,6975866,1137150,8113016
2,completo + few-shot,12696,SI,14059779,1137150,15196929


DIMENSIONAMIENTO DE LA MUESTRA (Gemini free tier)
  Config de produccion: ~10,099 tok/llamada -> SI (TPM 250,000)
  RPD 1,000  /  hasta 3 llamadas por conversacion  ->  ~333 conversaciones/dia

  muestra de  100  ->  ~0.3 dias
  muestra de  200  ->  ~0.6 dias
  muestra de  300  ->  ~0.9 dias
  muestra de  400  ->  ~1.2 dias
  muestra de 1197  ->  ~3.6 dias   <- corpus completo

  Recomendado: muestra ESTRATIFICADA de ~300 (por tramo de longitud y
  presencia de AGENTE), procesable en ~1 dia. Los agregados se reportan
  con su margen de error; para una prueba tecnica es defendible.


In [10]:
# Costo: la celda NO asume precios. Proyecta sobre una rejilla para que usted
# ubique el suyo si el proveedor cobra. Gemini AI Studio free tier es gratuito.
entrada_M = escenarios.iloc[0].tokens_entrada / 1e6
salida_M = escenarios.iloc[0].tokens_salida / 1e6

rejilla = []
for p_in in [0.00, 0.05, 0.10, 0.15, 0.20, 0.50]:
    for p_out in [0.00, 0.20, 0.50, 0.75]:
        rejilla.append({
            "USD/M entrada": p_in,
            "USD/M salida": p_out,
            "Costo corrida (USD)": round(entrada_M * p_in + salida_M * p_out, 3),
        })
tabla_costo = pd.DataFrame(rejilla).pivot(
    index="USD/M entrada", columns="USD/M salida", values="Costo corrida (USD)"
)
print(f"Volumen: {entrada_M:.2f} M tokens de entrada, {salida_M:.2f} M de salida")
print("\nCosto total de la corrida completa segun el precio por millon de tokens (USD):")
display(tabla_costo)
print("En el free tier de Gemini el costo monetario es 0; la restriccion son las peticiones/dia (RPD), no el precio.")

Volumen: 10.95 M tokens de entrada, 1.14 M de salida

Costo total de la corrida completa segun el precio por millon de tokens (USD):


USD/M salida,0.00,0.20,0.50,0.75
USD/M entrada,,,,
0.00,0.000,0.227,0.569,0.853
0.05,0.548,0.775,1.116,1.400
0.10,1.095,1.323,1.664,1.948
0.15,1.643,1.870,2.211,2.496
0.20,2.190,2.418,2.759,3.043
0.50,5.476,5.703,6.044,6.329


En el free tier de Gemini el costo monetario es 0; la restriccion son las peticiones/dia (RPD), no el precio.


---
## 3. Motor de extracción

Cuatro problemas prácticos que hay que resolver para que una corrida de 1.197 llamadas termine sin intervención manual:

**a. El catálogo de modelos del proveedor cambia.** Modelos que funcionaban se deprecan. Por eso hay una celda que consulta `client.models.list()` en vez de dar por hecho que `gpt-oss-120b` sigue disponible.

**b. Un modelo puede envolver el JSON.** Algunos modelos generan razonamiento o envuelven la respuesta en un bloque de markdown y `json.loads` falla. Se mantiene la limpieza defensiva de la respuesta, que quita las comillas triples y cualquier `<think>...</think>` antes de `json.loads`.

**c. Una respuesta puede ser JSON válido pero violar la regla de negocio.** Ahí es donde entran los validadores de `esquema.py`. El reintento no repite la misma petición, eso reproduce el mismo fallo: le **devuelve al modelo el mensaje de error** para que se corrija. Los mensajes de error de `esquema.py` están redactados como instrucciones dirigidas al modelo precisamente para esto.

**d. El 429 es normal, no excepcional.** Con límites por minuto y por segundo, chocar contra la cuota es parte de la operación. Se espera el `retry-after` que envía el servidor cuando viene, y si no, un respaldo largo.

**e. La salida se puede forzar en la petición.** Cuando el modelo lo soporta (`gpt-oss`, instruct de Mistral, `llama-4`), `_llamar_api` pasa `response_format` con el JSON Schema derivado de `esquema.py` (`prompts.formato_respuesta()`), que restringe las taxonomías a sus valores válidos. Gemini, el proveedor original, rechaza `$defs`/`$ref` en el schema, así que ahí se usa directamente `json_object`: JSON válido garantizado, y el contrato lo imponen la validación de Pydantic y el reintento dirigido. Con OpenAI el `json_schema` estructurado sí funciona, pero la corrida final también usa `json_object` a propósito (bitácora v13): el schema estricto de `esquema.py` lleva `$defs`/`$ref`/`default`, y el modo estricto de OpenAI es quisquilloso con eso — mejor la ruta ya probada en el piloto. La poda de las restricciones de longitud del schema hace que una respuesta pueda pasar el `json_schema` y aun así violar a Pydantic (resumen corto, evidencia larga), así que el reintento con `mensaje_de_reintento` sigue siendo la red final.

In [11]:
from dotenv import load_dotenv
load_dotenv(RAIZ / ".env")

import openai

# --- Proveedor del LLM. Cambiar SOLO esta linea para migrar. ---------------
# Los tres exponen API compatible con OpenAI; el resto del motor no cambia.
PROVEEDOR = "openai"   # "openai" (de pago, corpus completo) | "gemini" (gratis) | "cerebras"/"mistral"

_PROVEEDORES = {
    "openai": {
        "env": "OPENAI_API_KEY",
        "base_url": "https://api.openai.com/v1",
        "modelo": "gpt-4o-mini",             # v13: nano sub-extraia motivo_no_pago; 4o-mini clasifica mejor (no reasoning)
        "seed_param": "seed",                # gpt-4.1-* aceptan seed; _llamar_api lo quita si el server lo rechaza
        "json_schema": False,                # el schema estricto lleva $defs/$ref/default -> json_object + Pydantic + reintento (probado en el piloto)
    },
    "gemini": {
        "env": "GEMINI_API_KEY",
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "modelo": "gemini-3.5-flash-lite",   # 2.5-flash-lite deprecado; alt: "gemini-2.5-flash" / "gemini-3.5-flash"
        "seed_param": None,                   # el endpoint compat de Gemini no acepta semilla
        "json_schema": False,                 # Gemini rechaza $defs/$ref -> json_object + Pydantic
    },
    "cerebras": {
        "env": "CEREBRAS_API_KEY", "base_url": "https://api.cerebras.ai/v1",
        "modelo": "gpt-oss-120b", "seed_param": "seed", "json_schema": True,
    },
    "mistral": {
        "env": "MISTRAL_API_KEY", "base_url": "https://api.mistral.ai/v1",
        "modelo": "mistral-medium-latest", "seed_param": "random_seed", "json_schema": True,
    },
}
_CFG = _PROVEEDORES[PROVEEDOR]

API_KEY = os.environ.get(_CFG["env"])
if not API_KEY:
    raise RuntimeError(
        f"Falta {_CFG['env']}. Definala en el .env de la raiz como {_CFG['env']}=su_clave"
    )

cliente = openai.OpenAI(api_key=API_KEY, base_url=_CFG["base_url"])
MODELO = _CFG["modelo"]
SEED_PARAM = _CFG["seed_param"]
MODELO_SOPORTA_JSON_SCHEMA = _CFG["json_schema"]

print(f"Proveedor: {PROVEEDOR}  |  modelo: {MODELO}  |  {_CFG['base_url']}")
print("Salida estructurada (json_schema):", MODELO_SOPORTA_JSON_SCHEMA,
      "(si es False: json_object + validacion Pydantic + reintento dirigido)")

Proveedor: gemini  |  modelo: gemini-3.5-flash-lite  |  https://generativelanguage.googleapis.com/v1beta/openai/
Salida estructurada (json_schema): False (si es False: json_object + validacion Pydantic + reintento dirigido)


In [12]:
# El catalogo del proveedor cambia: verificar que el modelo objetivo existe.
try:
    catalogo = pd.DataFrame([
        {"id": m.id,
         "owned_by": getattr(m, "owned_by", None),
         "context_window": getattr(m, "context_window",
                                   getattr(m, "max_context_length", None))}
        for m in cliente.models.list().data
    ]).sort_values("id")
    display(catalogo)
    disponible = MODELO in set(catalogo.id) or f"models/{MODELO}" in set(catalogo.id)
    print(f"\n{MODELO} disponible: {disponible}")
    if not disponible:
        print("ATENCION: el modelo objetivo no aparece en el catalogo. "
              "Elija un reemplazo de la lista antes de continuar.")
except Exception as e:
    print("No se pudo consultar el catalogo:", type(e).__name__, e)

,id,owned_by,context_window
36,models/antigravity-preview-05-2026,google,None
43,models/aqa,google,None
37,models/deep-research-max-preview-04-2026,google,None
38,models/deep-research-preview-04-2026,google,None
39,models/deep-research-pro-preview-12-2025,google,None
35,models/gemini-2.5-computer-use-preview-10-2025,google,None
0,models/gemini-2.5-flash,google,None
10,models/gemini-2.5-flash-image,google,None
9,models/gemini-2.5-flash-lite,google,None
48,models/gemini-2.5-flash-native-audio-latest,google,None



gemini-3.5-flash-lite disponible: True


In [13]:
# --- Limpieza defensiva de la respuesta ----------------------------------
_RE_THINK = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
_RE_THINK_ABIERTO = re.compile(r"<think>.*$", re.DOTALL | re.IGNORECASE)
_RE_FENCE = re.compile(r"^\s*```(?:json)?\s*|\s*```\s*$", re.IGNORECASE)


def limpiar_respuesta(texto: str) -> str:
    """Deja solo el objeto JSON: quita razonamiento y envoltura de markdown."""
    t = _RE_THINK.sub("", texto)
    t = _RE_THINK_ABIERTO.sub("", t)      # bloque <think> sin cerrar por truncamiento
    t = _RE_FENCE.sub("", t.strip())
    t = t.strip()
    # Ultimo recurso: recortar al primer '{' y su llave de cierre correspondiente
    if not t.startswith("{"):
        i = t.find("{")
        if i >= 0:
            t = t[i:]
    if not t.endswith("}"):
        j = t.rfind("}")
        if j >= 0:
            t = t[: j + 1]
    return t


# Prueba con casos sinteticos: no requiere API
_casos = [
    '{"a": 1}',
    '<think>Analicemos...</think>\n{"a": 1}',
    '```json\n{"a": 1}\n```',
    'Aqui esta el resultado:\n{"a": 1}\nEspero que sirva.',
    '<think>razonando sin cerrar la etiqueta',
]
for c in _casos:
    salida = limpiar_respuesta(c)
    ok = salida == '{"a": 1}' or salida == ""
    print(f"  {'OK ' if ok else 'REV'}  {c[:45]!r:<50} -> {salida!r}")

  OK   '{"a": 1}'                                         -> '{"a": 1}'
  OK   '<think>Analicemos...</think>\n{"a": 1}'           -> '{"a": 1}'
  OK   '```json\n{"a": 1}\n```'                           -> '{"a": 1}'
  OK   'Aqui esta el resultado:\n{"a": 1}\nEspero que s'  -> '{"a": 1}'
  OK   '<think>razonando sin cerrar la etiqueta'          -> ''


In [14]:
CACHE_COMPLETA = DIR_OUT / "cache_extraccion.jsonl"
CACHE_MUESTRA = DIR_OUT / "cache_muestra.jsonl"
CACHE_PILOTO = DIR_OUT / "cache_piloto.jsonl"
CACHE_ESTABILIDAD = DIR_OUT / "cache_estabilidad.jsonl"


def cargar_cache(ruta: Path) -> dict:
    """Lee el JSONL y devuelve {uk_id_conversacion: registro}.

    Una linea corrupta (por ejemplo, si el proceso se interrumpio a mitad de
    escritura) se ignora en vez de tumbar la carga.
    """
    if not Path(ruta).exists():
        return {}
    registros = {}
    with open(ruta, "r", encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()
            if not linea:
                continue
            try:
                r = json.loads(linea)
                registros[r["uk_id_conversacion"]] = r
            except json.JSONDecodeError:
                continue
    return registros


def anexar_cache(ruta: Path, registro: dict) -> None:
    """Escribe una linea y hace flush inmediato.

    El flush importa: sin el, una interrupcion pierde todo lo que quedo en el
    buffer, es decir, llamadas ya pagadas.
    """
    with open(ruta, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")
        f.flush()


print("Estado de las caches:")
for nombre, ruta in [("completa", CACHE_COMPLETA), ("muestra", CACHE_MUESTRA), ("piloto", CACHE_PILOTO), ("estabilidad", CACHE_ESTABILIDAD)]:
    print(f"  {nombre:<12} {len(cargar_cache(ruta)):>5} registros   ({ruta.name})")

Estado de las caches:
  completa         0 registros   (cache_extraccion.jsonl)
  piloto          15 registros   (cache_piloto.jsonl)
  estabilidad      0 registros   (cache_estabilidad.jsonl)


In [15]:
MAX_REINTENTOS_VALIDACION = prompts.MAX_REINTENTOS
ESPERA_429_POR_DEFECTO = 65      # segundos; los limites del proveedor son por minuto
MAX_REINTENTOS_RED = 4


def _llamar_api(mensajes: list, semilla: int = 42, temperatura=None):
    """Una llamada al modelo, con manejo de 429 y de errores transitorios.

    - Parametros base en prompts.PARAMS_EXTRACCION (temperature 0). 'temperatura'
      los sobreescribe para el muestreo de autoconsistencia (seccion 7.a).
    - response_format: json_schema si MODELO_SOPORTA_JSON_SCHEMA; si el servidor
      lo rechaza, se cae a json_object (Gemini va directo a json_object).
    - La semilla se pasa con el nombre del proveedor (SEED_PARAM): 'seed' en
      Cerebras, 'random_seed' via extra_body en Mistral, None en Gemini. Reduce
      la varianza, no da determinismo; si el servidor la rechaza se quita.
    - 401/402/403: NO se reintenta. Es un problema de key o de plan/billing del
      proveedor; se corta con un mensaje claro.
    """
    parametros = dict(
        model=MODELO,
        messages=mensajes,
        **prompts.PARAMS_EXTRACCION,
    )
    if SEED_PARAM == "seed":
        parametros["seed"] = semilla
    elif SEED_PARAM == "random_seed":
        parametros["extra_body"] = {"random_seed": semilla}
    if temperatura is not None:
        parametros["temperature"] = temperatura
    parametros["response_format"] = prompts.formato_respuesta(
        estructurado=MODELO_SOPORTA_JSON_SCHEMA
    )
    for intento in range(MAX_REINTENTOS_RED):
        try:
            r = cliente.chat.completions.create(**parametros)
            return r.choices[0].message.content, r.usage
        except openai.BadRequestError as e:
            msg = str(e).lower()
            if ("seed" in msg) and ("seed" in parametros or "extra_body" in parametros):
                parametros.pop("seed", None)
                parametros.pop("extra_body", None)
                continue
            if (parametros["response_format"].get("type") == "json_schema" and
                    any(t in msg for t in ("response_format", "json_schema", "schema", "json_validate"))):
                print("    json_schema rechazado -> se reintenta con json_object")
                parametros["response_format"] = {"type": "json_object"}
                continue
            raise
        except openai.RateLimitError as e:
            espera = ESPERA_429_POR_DEFECTO
            cabeceras = getattr(getattr(e, "response", None), "headers", None) or {}
            for clave in ("retry-after", "x-ratelimit-reset-tokens", "x-ratelimit-reset-requests"):
                valor = cabeceras.get(clave)
                if valor:
                    try:
                        espera = max(espera, float(str(valor).rstrip("smh")) + 2)
                    except ValueError:
                        pass
                    break
            print(f"    429: cuota agotada, esperando {espera:.0f}s...")
            time.sleep(espera)
        except openai.APIStatusError as e:
            code = getattr(e, "status_code", None)
            if code in (401, 402, 403):
                raise RuntimeError(
                    f"El proveedor '{PROVEEDOR}' rechazo la peticion (HTTP {code}): "
                    f"{getattr(e, 'message', str(e))[:200]}. "
                    "Revisa la API key en .env o el plan/billing del proveedor. "
                    "No se reintenta."
                ) from None
            if code and code >= 500:
                espera = 2 ** intento
                print(f"    HTTP {code}: reintento en {espera}s")
                time.sleep(espera)
                continue
            raise
        except (openai.APIConnectionError, openai.APITimeoutError) as e:
            espera = 2 ** intento
            print(f"    {type(e).__name__}: reintento en {espera}s")
            time.sleep(espera)
    raise RuntimeError(f"Agotados {MAX_REINTENTOS_RED} intentos de red")


def extraer_conversacion(cid: str, transcripcion: str, semilla: int = 42) -> dict:
    """Extrae las variables de una conversacion y devuelve el registro a cachear.

    Flujo: llamada -> limpieza -> json.loads -> validacion Pydantic. Si la
    validacion falla, se le devuelve el error al modelo como turno de usuario
    para que corrija, hasta MAX_REINTENTOS_VALIDACION veces.

    Nunca lanza excepcion por un fallo del modelo: devuelve un registro con
    ok=False. Una conversacion problematica no debe tumbar una corrida de 1.197.
    """
    mensajes = prompts.construir_mensajes(transcripcion, compacto=True, con_few_shot=True)
    tokens_usados, ultimo_error = 0, None

    for intento in range(MAX_REINTENTOS_VALIDACION + 1):
        crudo, uso = _llamar_api(mensajes, semilla=semilla)
        if uso is not None:
            tokens_usados += getattr(uso, "total_tokens", 0) or 0
        texto = limpiar_respuesta(crudo or "")

        try:
            datos = json.loads(texto)
        except json.JSONDecodeError as e:
            ultimo_error = f"La respuesta no es JSON valido: {e}"
        else:
            try:
                objeto = ConversacionIA.model_validate(datos)
                return {
                    "uk_id_conversacion": cid,
                    "ok": True,
                    "intentos": intento + 1,
                    "tokens": tokens_usados,
                    "datos": objeto.model_dump(mode="json"),
                }
            except Exception as e:
                ultimo_error = str(e)

        if intento < MAX_REINTENTOS_VALIDACION:
            mensajes = mensajes + [
                {"role": "assistant", "content": texto[:4000]},
                prompts.mensaje_de_reintento(ultimo_error),
            ]

    return {
        "uk_id_conversacion": cid,
        "ok": False,
        "intentos": MAX_REINTENTOS_VALIDACION + 1,
        "tokens": tokens_usados,
        "error": ultimo_error,
        "datos": None,
    }


def procesar(ids, ruta_cache: Path, semilla: int = 42, pausa: float = 0.0, etiqueta: str = ""):
    """Procesa una lista de conversaciones respetando la cache incremental."""
    hechas = cargar_cache(ruta_cache)
    pendientes = [c for c in ids if c not in hechas]
    print(f"{etiqueta}{len(ids)} solicitadas | {len(ids)-len(pendientes)} en cache | {len(pendientes)} por procesar")

    t0 = time.time()
    for i, cid in enumerate(pendientes, 1):
        reg = extraer_conversacion(cid, transcripciones[cid], semilla=semilla)
        anexar_cache(ruta_cache, reg)
        estado = "ok" if reg["ok"] else f"FALLO: {str(reg.get('error'))[:70]}"
        print(f"  [{i:>4}/{len(pendientes)}] {cid[:28]:<28} {estado}")
        if pausa:
            time.sleep(pausa)

    if pendientes:
        print(f"\nTiempo: {(time.time()-t0)/60:.1f} min para {len(pendientes)} conversaciones")
    return cargar_cache(ruta_cache)


print("Motor de extraccion definido. Ninguna llamada realizada todavia.")

Motor de extraccion definido. Ninguna llamada realizada todavia.


---
## 4. Ejecución piloto, 15 conversaciones

El piloto se ejecuta **antes** que la Ejecución completa y con su propia caché, para no contaminar los resultados definitivos. Su función no es producir datos sino responder cuatro preguntas antes de comprometer la cuota:

1. ¿El modelo devuelve JSON parseable de forma consistente?
2. ¿Cuántas respuestas necesitan reintento por validación? Si es una fracción alta, el problema está en el prompt y hay que corregirlo antes, no después.
3. ¿El consumo real de tokens por llamada coincide con lo estimado en la sección 2?
4. Leyendo las 15 salidas contra sus transcripciones, ¿las categorías y las evidencias tienen sentido?

La muestra no es aleatoria simple: se estratifica por longitud y por presencia de asesor humano, para que el piloto incluya los casos difíciles  sin `AGENTE`, conversaciones muy largas— y no sólo los cómodos.

In [16]:
rng = np.random.default_rng(42)

perfil = (limpio.groupby("uk_id_conversacion")
                .agg(n_mensajes=("uk_id_mensaje", "size"),
                     n_agente=("de", lambda s: (s == "AGENTE").sum()),
                     n_usuario=("de", lambda s: (s == "USUARIO").sum()))
                .reset_index())
perfil["tramo"] = pd.qcut(perfil.n_mensajes, 3, labels=["corta", "media", "larga"])
perfil["sin_agente"] = perfil.n_agente == 0

N_PILOTO = 15

# Casos dificiles primero: las conversaciones sin ningun AGENTE humano son las
# que ponen a prueba la regla de roles, asi que entran todas las que haya.
piloto_ids = list(perfil.loc[perfil.sin_agente, "uk_id_conversacion"])

# Luego 4 por tramo de longitud, para cubrir cortas, medias y largas
for tramo, g in perfil.groupby("tramo", observed=True):
    disponibles = g[~g.uk_id_conversacion.isin(piloto_ids)]
    piloto_ids += list(disponibles.sample(min(4, len(disponibles)), random_state=42).uk_id_conversacion)

# Si aun falta para llegar a 15, se completa al azar sobre el resto
piloto_ids = list(dict.fromkeys(piloto_ids))
if len(piloto_ids) < N_PILOTO:
    resto = perfil[~perfil.uk_id_conversacion.isin(piloto_ids)]
    piloto_ids += list(resto.sample(N_PILOTO - len(piloto_ids), random_state=42).uk_id_conversacion)
piloto_ids = piloto_ids[:N_PILOTO]

print(f"Piloto: {len(piloto_ids)} conversaciones")
display(perfil[perfil.uk_id_conversacion.isin(piloto_ids)]
        [["uk_id_conversacion", "n_mensajes", "n_agente", "tramo", "sin_agente"]])

Piloto: 15 conversaciones


,uk_id_conversacion,n_mensajes,n_agente,tramo,sin_agente
37,CONV_00000038,40,12,larga,False
85,CONV_00000086,26,11,media,False
98,CONV_00000099,64,22,larga,False
198,CONV_00000199,26,7,media,False
307,CONV_00000308,23,6,corta,False
324,CONV_00000325,27,8,media,False
352,CONV_00000353,22,0,corta,True
477,CONV_00000478,22,4,corta,False
517,CONV_00000518,60,23,larga,False
618,CONV_00000619,27,7,media,False


In [35]:
# EJECUTABLE. Consume cuota del proveedor: ~15 llamadas.
resultados_piloto = procesar(piloto_ids, CACHE_PILOTO, pausa=0.3, etiqueta="Piloto: ")  # <- HECHO 2026-09-10: 15/15 validas, cache poblada
print("Celda del piloto lista. Descomente la linea para ejecutarla.")

Piloto: 15 solicitadas | 15 en cache | 0 por procesar
Celda del piloto lista. Descomente la linea para ejecutarla.


In [36]:
# Diagnostico del piloto: solo corre si la cache tiene registros.
piloto = cargar_cache(CACHE_PILOTO)
if not piloto:
    print("Cache del piloto vacia. Ejecute la celda anterior para poblarla.")
else:
    ok = [r for r in piloto.values() if r["ok"]]
    fallos = [r for r in piloto.values() if not r["ok"]]
    print(f"Procesadas      : {len(piloto)}")
    print(f"Validas         : {len(ok)} ({len(ok)/len(piloto)*100:.0f}%)")
    print(f"Fallidas        : {len(fallos)}")
    print(f"Con reintento   : {sum(1 for r in ok if r['intentos'] > 1)}")
    if ok:
        toks = [r["tokens"] for r in ok if r["tokens"]]
        if toks:
            print(f"Tokens/llamada  : media {np.mean(toks):,.0f} | max {np.max(toks):,.0f}")
            print(f"  (estimacion de la seccion 2: ~{tok_prompt_fijo + tok_transcripciones/n_conv + TOKENS_SALIDA_ESTIMADOS:,.0f})")
    for r in fallos[:5]:
        print(f"  FALLO {r['uk_id_conversacion']}: {str(r.get('error'))[:150]}")

Procesadas      : 15
Validas         : 15 (100%)
Fallidas        : 0
Con reintento   : 7
Tokens/llamada  : media 16,749 | max 26,542
  (estimacion de la seccion 2: ~10,099)


In [37]:
# Revision cualitativa: la salida del modelo junto a su transcripcion.
# Es el paso que ninguna metrica reemplaza; leer 15 casos revela problemas de
# prompt que ningun agregado muestra.
if piloto:
    r = next(iter([x for x in piloto.values() if x["ok"]]), None)
    if r:
        cid = r["uk_id_conversacion"]
        print("TRANSCRIPCION (primeros 1200 caracteres)")
        print("=" * 100)
        print(transcripciones[cid][:1200])
        print()
        print("VARIABLES EXTRAIDAS")
        print("=" * 100)
        print(json.dumps(r["datos"], ensure_ascii=False, indent=2))

TRANSCRIPCION (primeros 1200 caracteres)
[HSM] BANCO_000001 :BLUE_HEART: ¡HOLA!  HEMOS INTENTADO COMUNICARNOS CONTIGO PARA HABLAR SOBRE EL ESTADO DE TUS PRODUCTOS FINANCIEROS. TENEMOS OPCIONES PARA QUE LOGRES NORMALIZAR TUS PAGOS Y RECUPERES TU TRANQUILIDAD FINANCIERA.  EVITA RECARGOS ADICIONALES Y MANTÉN TUS PRODUCTOS AL DÍA. ¿PODEMOS APOYARTE AHORA?'
[USUARIO] DISCULPE
[BOT] => SÍ, DESEO CONTACTARME (SÍ, DESEO CONTACTARME), NO ES UN BUEN MOMENTO (NO ES UN BUEN MOMENTO), REALIZAR EL PAGO (REALIZAR EL PAGO)
[USUARIO] PERO QUIERO VER LA CUENTA
[BOT] => SÍ, DESEO CONTACTARME (SÍ, DESEO CONTACTARME), NO ES UN BUEN MOMENTO (NO ES UN BUEN MOMENTO), REALIZAR EL PAGO (REALIZAR EL PAGO)
[USUARIO] SÍ, DESEO CONTACTARME
[BOT] AL CONTINUAR NOS AUTORIZAS A QUE TUS DATOS SEAN TRATADOS CONFORME A LA LEY 1581 DEL FECHA_00000213 DE PROTECCIÓN DE DATOS QUE SE ENCUENTRA EN EL SIGUIENTE ENLACE: URL_ANONIMIZADA
[BOT] ESTAMOS VALIDANDO TU INFORMACIÓN. EN UN MOMENTO, UNO DE NUESTROS ASESORES TE AYUDARÁ A EN

---
## 5. Corrida completa

La corrida real de este proyecto no se lanzó descomentando una línea acá: por
el volumen —1.197 conversaciones, horas de ejecución— se disparó en segundo
plano con `notebooks/_runner_extraccion.py`, que ejecuta el mismo código que
las celdas de arriba y llama a `procesar()` sobre `todos_ids`. La línea de
`procesar()` de la celda 27 queda comentada a propósito: si alguien la corre a
mano mientras el runner sigue vivo, los dos procesos terminan escribiendo la
misma caché y pisándose el uno al otro.

El camino hasta el resultado final tuvo dos paradas, no una:

- **Con `gpt-4.1-nano`**, la corrida completó las 1.197 sin errores de
  ejecución, pero una auditoría posterior mostró que el modelo sub-extraía el
  motivo de no pago casi siempre.
- **Con `gpt-4o-mini`**, después de tapar también un hueco en la validación
  del acuerdo de pago, la cobertura llegó a 100% (0 fallos) y la calidad de
  `motivo_no_pago` mejoró de forma sustancial.

Los detalles de qué se encontró y por qué se cambió de modelo están en la
sección de metodología, al principio del notebook.

La corrida es interrumpible en cualquiera de los dos casos: cada resultado se
guarda apenas se obtiene, así que si se corta basta con volver a lanzarla —
lo ya procesado no se repite ni se vuelve a pagar.

In [20]:
# ============================================================================
# CORRIDA DE PRODUCCION -- CORPUS COMPLETO con OpenAI de pago (gpt-4.1-nano).
# Con proveedor de pago ya no aplica el RPD del free tier: se procesan las
# 1.197 conversaciones (~$3-4). El muestreo estratificado de abajo se conserva
# como documentacion del diseno muestral y como respaldo si se quisiera acotar.
# Los agregados de las preguntas a-e se reportan sobre el corpus completo.
#   - Cache incremental CACHE_COMPLETA: la corrida es interrumpible y resumible.
#   - La dispara notebooks/_runner_extraccion.py en segundo plano (no esta celda).
# ============================================================================
N_MUESTRA = 300
CACHE_MUESTRA = DIR_OUT / "cache_muestra.jsonl"

# perfil viene de la celda 22: n_mensajes, tramo (corta/media/larga), sin_agente.
# En este corpus solo 1 conversacion no tiene ningun AGENTE humano, asi que el
# eje real de estratificacion es el tramo de longitud; se conserva sin_agente
# para que esa unica conversacion entre siempre en la muestra.
estratos = perfil.groupby(["tramo", "sin_agente"], observed=True)
pesos_estrato = estratos.size()

# Asignacion proporcional al tamano del estrato, minimo 5, tope = lo disponible.
asignacion = (pesos_estrato / pesos_estrato.sum() * N_MUESTRA).round().astype(int)
asignacion = asignacion.clip(lower=5)
asignacion = np.minimum(asignacion, pesos_estrato)

muestra_ids = []
for (_tramo, _sin_ag), g in estratos:
    k = int(asignacion.loc[(_tramo, _sin_ag)])
    muestra_ids += list(g.sample(k, random_state=2026).uk_id_conversacion)
muestra_ids = list(dict.fromkeys(muestra_ids))

_dist = (perfil[perfil.uk_id_conversacion.isin(muestra_ids)]
         .groupby(["tramo", "sin_agente"], observed=True).size().rename("en_muestra"))
print(f"Muestra estratificada: {len(muestra_ids)} de {len(perfil):,} conversaciones")
print(pd.concat([pesos_estrato.rename("en_corpus"), _dist], axis=1)
      .fillna(0).astype(int).to_string())

# --- Corrida real: CORPUS COMPLETO -> CACHE_COMPLETA -----------------------
# EJECUTABLE, consume saldo de OpenAI. La lanza en segundo plano
# notebooks/_runner_extraccion.py (ejecuta el source de las celdas de setup +
# este muestreo y luego procesar(todos_ids, CACHE_COMPLETA)). No descomentar
# aqui mientras el runner corra: dos procesos sobre la misma cache se pisan.
# resultados = procesar(todos_ids, CACHE_COMPLETA, pausa=2.0, etiqueta="Corpus: ")
todos_ids = list(tr.uk_id_conversacion)
print(f"Corrida de produccion: corpus completo = {len(todos_ids):,} conversaciones")
print(f"  destino: {CACHE_COMPLETA.name}  |  ya en cache: {len(cargar_cache(CACHE_COMPLETA))}")
print(f"  (muestra_ids = {len(muestra_ids)} queda como referencia del diseno muestral)")


Corrida completa NO ejecutada (celda comentada).
Al descomentarla procesaria 1,197 conversaciones.
Ya hay 0 registros en cache; esas se saltarian.


---
## 6. Consolidación

Se producen cuatro tablas con dos formas distintas, porque responden a preguntas distintas:

**Formato ancho — una fila por conversación** (`conversaciones_ia.csv`). Es la tabla de hechos: sirve para conteos, para el score de satisfacción y para el ranking de la pregunta (e).

**Formato largo — una fila por conversación-elemento** (`ofertas_largo.csv`, `argumentos_largo.csv`, `factores_largo.csv`). Una conversación puede contener varias ofertas, y meterlas en una celda separadas por comas hace imposible cualquier agregación. El formato largo es además lo que Power BI espera para construir la relación 1:N y permitir filtrado cruzado sin trucos de DAX.

La pregunta (c) —*qué ofrecimientos logran más acuerdos*— se responde exactamente uniendo `ofertas_largo` con `acuerdo_pago` de la tabla ancha y calculando la tasa de conversión por tipo de oferta. La estructura de estas tablas está pensada para eso.

In [21]:
def consolidar(cache: dict):
    """Convierte la cache en cuatro tablas: ancha + tres largas."""
    filas, ofertas, argumentos, factores = [], [], [], []

    # Mensajes del rol USUARIO por conversacion, para verificar que la cita de
    # acuerdo_evidencia la dijo el cliente y no el asesor (hueco de REGLA_ACUERDO
    # que Pydantic no tapa). Ver esquema.acuerdo_confirmado_por_cliente.
    _msgs_usuario = (limpio[limpio.de == "USUARIO"]
                     .groupby("uk_id_conversacion")["msg"].apply(list).to_dict())

    for cid, reg in cache.items():
        if not reg.get("ok") or not reg.get("datos"):
            continue
        d = reg["datos"]
        _acuerdo_verificado = bool(d["acuerdo_pago"]) and esquema.acuerdo_confirmado_por_cliente(
            d["acuerdo_evidencia"], _msgs_usuario.get(cid, [])
        )
        # v13: un motivo solo vale si su evidencia es una frase textual del cliente.
        _motivo_llm = d["motivo_no_pago"]
        _motivo_ok = _motivo_llm != "NINGUNO" and esquema.motivo_respaldado_por_cliente(
            d["evidencia_motivo"], _msgs_usuario.get(cid, [])
        )
        _motivo_final = _motivo_llm if _motivo_ok else "NINGUNO"

        filas.append({
            "uk_id_conversacion": cid,
            "resumen": d["resumen"],
            "motivo_no_pago": _motivo_final,
            "motivo_no_pago_llm": _motivo_llm,
            "motivo_reclasificado": _motivo_llm != "NINGUNO" and not _motivo_ok,
            "evidencia_motivo": d["evidencia_motivo"],
            "n_motivos_secundarios": len(d["motivos_secundarios"]),
            "motivos_secundarios": "|".join(d["motivos_secundarios"]),
            "cliente_reconoce_deuda": d["cliente_reconoce_deuda"],
            "asesor_pregunta_por_motivo": d["asesor_pregunta_por_motivo"],
            "n_ofertas": len(d["ofertas_asesor"]),
            "tipos_oferta": "|".join(o["tipo"] for o in d["ofertas_asesor"]),
            "n_argumentos": len(d["argumentos_asesor"]),
            "tipos_argumento": "|".join(a["tipo"] for a in d["argumentos_asesor"]),
            "acuerdo_pago": _acuerdo_verificado,
            "acuerdo_pago_llm": bool(d["acuerdo_pago"]),
            "acuerdo_reclasificado": bool(d["acuerdo_pago"]) and not _acuerdo_verificado,
            "fecha_compromiso_texto": d["fecha_compromiso_texto"],
            "tipo_fecha": d["tipo_fecha"],
            "monto_comprometido_texto": d["monto_comprometido_texto"],
            "acuerdo_evidencia": d["acuerdo_evidencia"],
            "satisfaccion_llm": d["satisfaccion_llm"],
            "justificacion_satisfaccion": d["justificacion_satisfaccion"],
            "n_factores_insatisfaccion": len([f for f in d["factores_insatisfaccion"] if f != "NINGUNO"]),
            "factores_insatisfaccion": "|".join(d["factores_insatisfaccion"]),
            "sentimiento_inicial": d["sentimiento_inicial"],
            "sentimiento_final": d["sentimiento_final"],
            "hubo_friccion_bot": d["hubo_friccion_bot"],
            "conversacion_abandonada": d["conversacion_abandonada"],
            "recomendacion_mejora": d["recomendacion_mejora"],
            "intentos_extraccion": reg.get("intentos"),
        })

        for i, o in enumerate(d["ofertas_asesor"]):
            ofertas.append({"uk_id_conversacion": cid, "orden": i + 1,
                            "tipo_oferta": o["tipo"], "evidencia": o["evidencia"]})
        for i, a in enumerate(d["argumentos_asesor"]):
            argumentos.append({"uk_id_conversacion": cid, "orden": i + 1,
                               "tipo_argumento": a["tipo"], "evidencia": a["evidencia"]})
        for i, f in enumerate(d["factores_insatisfaccion"]):
            if f != "NINGUNO":
                factores.append({"uk_id_conversacion": cid, "orden": i + 1, "factor": f})

    return (pd.DataFrame(filas), pd.DataFrame(ofertas),
            pd.DataFrame(argumentos), pd.DataFrame(factores))


# Se consolida sobre la mejor cache disponible: completa > muestra > piloto
cache_activa = (cargar_cache(CACHE_COMPLETA) or cargar_cache(CACHE_MUESTRA)
                or cargar_cache(CACHE_PILOTO))
conv_ia, ofertas_largo, argumentos_largo, factores_largo = consolidar(cache_activa)

print("Fuente:", "corrida completa" if cargar_cache(CACHE_COMPLETA)
      else "muestra estratificada" if cargar_cache(CACHE_MUESTRA) else "piloto")
print(f"  conversaciones_ia : {len(conv_ia):>6} filas x {len(conv_ia.columns) if len(conv_ia) else 0} columnas")
print(f"  ofertas_largo     : {len(ofertas_largo):>6} filas")
print(f"  argumentos_largo  : {len(argumentos_largo):>6} filas")
print(f"  factores_largo    : {len(factores_largo):>6} filas")
if conv_ia.empty:
    print("\nSin datos extraidos todavia. Las secciones siguientes quedaran vacias "
          "hasta que se ejecute el piloto o la corrida completa.")

if not conv_ia.empty:
    _rec = int(conv_ia.acuerdo_reclasificado.sum())
    print(f"\nAcuerdo de pago: {int(conv_ia.acuerdo_pago_llm.sum())} segun el LLM -> "
          f"{int(conv_ia.acuerdo_pago.sum())} confirmados por cita del USUARIO "
          f"({_rec} reclasificados a False por citar al asesor o no ser localizable)")
    _mrec = int(conv_ia.motivo_reclasificado.sum())
    _con_motivo = int((conv_ia.motivo_no_pago != "NINGUNO").sum())
    print(f"Motivo de no pago: {int((conv_ia.motivo_no_pago_llm != 'NINGUNO').sum())} con motivo segun el LLM -> "
          f"{_con_motivo} con evidencia textual del USUARIO ({_mrec} degradados a NINGUNO)")


Fuente: piloto
  conversaciones_ia :     15 filas x 27 columnas
  ofertas_largo     :     10 filas
  argumentos_largo  :     13 filas
  factores_largo    :     23 filas


In [22]:
# Vista previa de las respuestas a (a), (b) y (c). Vacia hasta que haya extraccion.
if not conv_ia.empty:
    print("(a) Motivos de no pago mas frecuentes")
    display(conv_ia.motivo_no_pago.value_counts().to_frame("conversaciones"))

    if not ofertas_largo.empty:
        print("\n(b) Ofertas mas realizadas por los asesores")
        display(ofertas_largo.tipo_oferta.value_counts().to_frame("veces"))

        print("\n(c) Efectividad por tipo de oferta (tasa de acuerdo)")
        # Se deduplica por (conversacion, tipo_oferta): si una conversacion repite
        # el mismo tipo de oferta, el acuerdo no debe contarse dos veces arriba.
        _pares = (ofertas_largo[["uk_id_conversacion", "tipo_oferta"]].drop_duplicates()
                  .merge(conv_ia[["uk_id_conversacion", "acuerdo_pago"]], on="uk_id_conversacion"))
        efect = (_pares.groupby("tipo_oferta")
                 .agg(conversaciones=("uk_id_conversacion", "nunique"),
                      acuerdos=("acuerdo_pago", "sum")))
        efect["tasa_acuerdo_%"] = (efect.acuerdos / efect.conversaciones * 100).round(1)
        display(efect.sort_values("tasa_acuerdo_%", ascending=False))
        print("ADVERTENCIA: es una tasa observacional, no un efecto causal. "
              "El asesor elige que ofrecer segun el caso, asi que la oferta esta "
              "confundida con la disposicion previa del cliente.")

(a) Motivos de no pago mas frecuentes


,conversaciones
motivo_no_pago,
DESACUERDO_MONTO_O_COBRO,2
NINGUNO,2
YA_PAGO_O_PAGO_NO_APLICADO,2
INGRESOS_INSUFICIENTES,2
NO_RECONOCE_LA_OBLIGACION,1
OLVIDO_O_DESCONOCIMIENTO,1
DESEMPLEO,1
ENFERMEDAD_O_CALAMIDAD,1
SOBREENDEUDAMIENTO,1



(b) Ofertas mas realizadas por los asesores


,veces
tipo_oferta,
REESTRUCTURACION_CREDITO,3
CANAL_PAGO_FACILITADO,2
PLAN_PAGO_DIFERIDO,2
PAGO_MINIMO_O_ABONO_PARCIAL,1
PRORROGA_O_APLAZAMIENTO,1
NORMALIZACION_AL_DIA,1



(c) Efectividad por tipo de oferta (tasa de acuerdo)


,conversaciones,acuerdos,tasa_acuerdo_%
tipo_oferta,,,
CANAL_PAGO_FACILITADO,2,2,100.0
NORMALIZACION_AL_DIA,1,1,100.0
PAGO_MINIMO_O_ABONO_PARCIAL,1,1,100.0
PLAN_PAGO_DIFERIDO,2,1,50.0
PRORROGA_O_APLAZAMIENTO,1,0,0.0
REESTRUCTURACION_CREDITO,3,0,0.0


ADVERTENCIA: es una tasa observacional, no un efecto causal. El asesor elige que ofrecer segun el caso, asi que la oferta esta confundida con la disposicion previa del cliente.


---
## 7. Score híbrido de satisfacción

> El enunciado exige entregar la *"Explicación de la metodología (que incluya prompts y **cálculo de la satisfacción**)"*. Esta sección es esa explicación.

### Por qué no basta con la calificación del LLM

Un LLM usado como juez tiene un sesgo bien documentado: **comprime sus calificaciones hacia el centro de la escala**. Tiende a repartir 3 y 4, y a evitar los extremos. Eso choca de frente con la pregunta (e), que pide *"las 5 conversaciones peor calificadas"*: si la cola baja está vacía, no hay nada que mostrar y el ranking se decide por empates arbitrarios.

Al mismo tiempo, sustituir el juicio del modelo por reglas mecánicas sería peor. Una conversación puede ser corta, sin repeticiones y con asesor humano, y aun así ser una mala experiencia porque el asesor fue cortante o no respondió lo que se le preguntó. Eso sólo se detecta leyendo.

### La combinación

$$\text{score} = 0{,}70 \times \text{componente LLM} + 0{,}30 \times \text{componente objetivo}$$

**Componente LLM** — la calificación 1-5 de la rúbrica anclada, reescalada a 0-100: $(s-1)/4 \times 100$.

**Componente objetivo** — parte de 100 y resta penalizaciones por señales verificables:

| Señal | Penalización | Por qué ese peso |
|---|---|---|
| Bucle del bot: mismo texto repetido 3+ veces | **−30** | La más grave y la más objetiva. Es un fallo del sistema, medible sin ambigüedad, y en el corpus va acompañado de clientes que escriben *"ya se los mandé dos veces"* |
| Sin ningún mensaje del `AGENTE` | **−25** | El cliente nunca fue atendido por un humano. Fallo total del canal |
| El cliente repite el mismo mensaje 2+ veces | **−15** | Señal de no haber sido escuchado. Menor que las anteriores porque puede deberse a un envío duplicado accidental |
| Longitud en el decil superior **y** sin acuerdo | **−15** | Conversación larga que no llega a nada. Va condicionada a *sin acuerdo* a propósito: una negociación larga que sí termina en compromiso es trabajo bien hecho, no fricción |
| Cierre en sentimiento negativo | **−15** | Cómo termina el cliente pesa más que cómo empieza |

Suma máxima de penalizaciones: 100. El componente se acota en $[0, 100]$.

### Por qué 70/30 y no otra proporción

El 70% reconoce que el LLM es la única fuente capaz de leer contenido y tono. El 30% es suficiente para que una conversación con varias señales de fricción caiga con claridad al fondo del ranking —una conversación con bucle de bot y sin agente pierde 55 de los 30 puntos objetivos, unos 16 puntos del score total— sin que una señal puramente mecánica pueda dominar el juicio.

Un reparto 50/50 dejaría que la longitud del texto pesara tanto como el contenido, lo que castigaría negociaciones largas y legítimas. Un 90/10 no corregiría la compresión al centro, que es el problema que motivó todo esto.

### Advertencia sobre la cuarta señal

Cuatro de las cinco señales se miden **directamente sobre los mensajes**, sin intervención del LLM. La quinta —cierre en sentimiento negativo— proviene de `sentimiento_final`, que **sí** genera el modelo. No es del todo independiente y hay que decirlo: aporta 15 de los 100 puntos objetivos, es decir 4,5 puntos del score final. Se conserva porque el sentimiento de cierre es el mejor predictor de experiencia disponible y ningún léxico de polaridad genérico funciona bien sobre este corpus en mayúsculas y con marcadores. La sección 7.b ofrece una verificación externa e independiente de todo el score.

### Los pesos son una hipótesis, no un resultado

Están fundamentados en el EDA y en la severidad relativa de cada fallo, pero **no están calibrados contra satisfacción real observada**. Es una escala de trabajo defendible, no una medida validada. La forma correcta de calibrarlos sería una regresión ordinal contra las anotaciones humanas de la sección 8 una vez existan, o contra el NPS de la sección 7.b.

**Una comprobación posterior (ver `docs/BRIEFING_RAG.md`) ya dio una señal en esa dirección, y conviene decirla aunque incomode:** al comparar contra el NPS declarado, la nota del LLM sola correlacionó mejor que el score híbrido. El componente objetivo sí logra lo que buscaba —bajar la dispersión y evitar que todo se apile en 3 y 4— pero le resta algo de concordancia con la única verdad de campo disponible. Para la presentación, lo honesto es mostrar ambos números por separado en vez de vender el híbrido como una mejora sin matices.

In [23]:
def senales_objetivas(limpio: pd.DataFrame, tr: pd.DataFrame) -> pd.DataFrame:
    """Calcula las senales medibles directamente sobre los mensajes."""
    g = limpio.groupby("uk_id_conversacion")

    s = pd.DataFrame({
        "n_mensajes": g.size(),
        "n_agente": g.de.apply(lambda x: (x == "AGENTE").sum()),
        "n_usuario": g.de.apply(lambda x: (x == "USUARIO").sum()),
    })

    # Bucle: maxima repeticion de un mismo texto por parte de BOT o AGENTE
    auto = limpio[limpio.de.isin(["BOT", "AGENTE"])]
    rep_auto = (auto.groupby(["uk_id_conversacion", "msg"]).size()
                    .groupby("uk_id_conversacion").max())
    s["max_repeticion_automatica"] = rep_auto.reindex(s.index).fillna(0).astype(int)

    # Repeticion del cliente: mensajes de 4+ caracteres, para no contar "SI"/"OK"
    usr = limpio[(limpio.de == "USUARIO") & (limpio.msg.str.len() >= 4)]
    rep_usr = (usr.groupby(["uk_id_conversacion", "msg"]).size()
                  .groupby("uk_id_conversacion").max())
    s["max_repeticion_cliente"] = rep_usr.reindex(s.index).fillna(0).astype(int)

    s = s.join(tr.set_index("uk_id_conversacion")[["n_caracteres"]])

    s["sen_bucle_bot"] = s.max_repeticion_automatica >= 3
    s["sen_sin_agente"] = s.n_agente == 0
    s["sen_cliente_repite"] = s.max_repeticion_cliente >= 2
    s["sen_decil_superior_longitud"] = s.n_caracteres >= s.n_caracteres.quantile(0.90)
    return s.reset_index()


senales = senales_objetivas(limpio, tr)

print("Prevalencia de las senales objetivas en el corpus completo (1.197 conversaciones):")
for c in ["sen_bucle_bot", "sen_sin_agente", "sen_cliente_repite", "sen_decil_superior_longitud"]:
    print(f"  {c:<32} {senales[c].sum():>5} conversaciones ({senales[c].mean()*100:>5.1f}%)")
print("\nNota: estas cuatro senales NO dependen del LLM. Se calculan sobre 1.197")
print("conversaciones aunque la extraccion aun no se haya ejecutado.")

Prevalencia de las senales objetivas en el corpus completo (1.197 conversaciones):
  sen_bucle_bot                      144 conversaciones ( 12.0%)
  sen_sin_agente                       1 conversaciones (  0.1%)
  sen_cliente_repite                 353 conversaciones ( 29.5%)
  sen_decil_superior_longitud        120 conversaciones ( 10.0%)

Nota: estas cuatro senales NO dependen del LLM. Se calculan sobre 1.197
conversaciones aunque la extraccion aun no se haya ejecutado.


In [24]:
PESOS_PENALIZACION = {
    "sen_bucle_bot": 30,
    "sen_sin_agente": 25,
    "sen_cliente_repite": 15,
    "sen_larga_sin_acuerdo": 15,
    "sen_cierre_negativo": 15,
}
PESO_LLM = 0.70
PESO_OBJETIVO = 0.30


def score_hibrido(conv_ia: pd.DataFrame, senales: pd.DataFrame) -> pd.DataFrame:
    """Combina la calificacion del LLM con las senales objetivas."""
    d = conv_ia.merge(senales, on="uk_id_conversacion", how="left")

    # Las dos senales que necesitan una variable del LLM
    d["sen_larga_sin_acuerdo"] = d.sen_decil_superior_longitud & (~d.acuerdo_pago.astype(bool))
    d["sen_cierre_negativo"] = d.sentimiento_final.isin(["NEGATIVO", "MUY_NEGATIVO"])

    d["componente_llm"] = (d.satisfaccion_llm - 1) / 4 * 100

    d["penalizacion_total"] = 0
    for senal, peso in PESOS_PENALIZACION.items():
        d["penalizacion_total"] += d[senal].fillna(False).astype(int) * peso
    d["componente_objetivo"] = (100 - d.penalizacion_total).clip(lower=0, upper=100)

    d["score_satisfaccion"] = (PESO_LLM * d.componente_llm
                               + PESO_OBJETIVO * d.componente_objetivo).round(1)
    d["senales_activas"] = d[list(PESOS_PENALIZACION)].fillna(False).astype(bool).sum(axis=1)
    return d


if not conv_ia.empty:
    conv_ia = score_hibrido(conv_ia, senales)
    print("Distribucion del score hibrido:")
    print(conv_ia.score_satisfaccion.describe().round(1).to_string())
    print()
    print("Comparacion de dispersion (el punto de todo el ejercicio):")
    print(f"  desv. est. solo LLM (0-100) : {conv_ia.componente_llm.std():.1f}")
    print(f"  desv. est. score hibrido    : {conv_ia.score_satisfaccion.std():.1f}")
    print()
    print("(e) Ranking preliminar de peor calificadas (se estabiliza en 7.a):")
    peores = conv_ia.nsmallest(5, "score_satisfaccion")
    display(peores[["uk_id_conversacion", "score_satisfaccion", "satisfaccion_llm",
                    "senales_activas", "factores_insatisfaccion", "recomendacion_mejora"]])
else:
    print("Sin extraccion todavia: el score se calculara cuando exista conv_ia.")

Distribucion del score hibrido:
count     15.0
mean      64.1
std       28.7
min       18.0
25%       43.0
50%       60.5
75%       89.0
max      100.0

Comparacion de dispersion (el punto de todo el ejercicio):
  desv. est. solo LLM (0-100) : 36.8
  desv. est. score hibrido    : 28.7

(e) Ranking preliminar de peor calificadas (se estabiliza en 7.a):


,uk_id_conversacion,score_satisfaccion,satisfaccion_llm,senales_activas,factores_insatisfaccion,recomendacion_mejora
0,CONV_00000353,18.0,1,2,SIN_RESPUESTA_HUMANA|NO_RESOLVIO_SOLICITUD|INFORMACION_CONTRADICTORIA|TRATO_INADECUADO,Implementar un protocolo de validacion y escalamiento inmediato cuando el usuario manifiesta desconocer el producto ...
1,CONV_00000478,25.5,1,1,NO_RESOLVIO_SOLICITUD|PROBLEMA_TECNICO_CANAL|INFORMACION_CONTRADICTORIA,Evitar que el bot dispare plantillas de cierre y encuesta de manera automatica mientras hay una conversacion activa ...
3,CONV_00001187,34.0,2,2,PROBLEMA_TECNICO_CANAL|NO_RESOLVIO_SOLICITUD|FALTA_DE_FLEXIBILIDAD_EN_LA_OFERTA,Ajustar la programacion del bot para que no interrumpa ni cierre por inactividad las conversaciones activas con un a...
4,CONV_00000308,43.0,2,1,PRESION_EXCESIVA|TRATO_INADECUADO,Evitar comunicaciones de cobro agresivas o tan tempranas para obligaciones con un solo dia de mora en clientes con b...
5,CONV_00000199,43.0,2,1,SOLICITUD_REPETIDA_DE_DATOS|NO_RESOLVIO_SOLICITUD|DEMORA_EXCESIVA,Mejorar la integracion de los extractos de libranza con el area de recaudo empresarial para evitar transferencias in...


---
## 7.a Autoconsistencia de la calificación

`satisfaccion_llm` es la única variable del ranking de la pregunta (e), y una sola llamada tiene varianza: dos conversaciones vecinas en la cola baja pueden intercambiar posición entre corridas. Para **ese campo y solo ese** se remuestrea.

**Diseño.** Se toman las `N` conversaciones peor calificadas del ranking preliminar y se piden `K` respuestas adicionales a `temperature` > 0 (`prompts.K_AUTOCONSISTENCIA`, `prompts.TEMP_AUTOCONSISTENCIA`). Sobre las `K+1` notas —las nuevas más la de la corrida a `temperature` 0— se toma la **mediana entera** con `prompts.consolidar_satisfaccion()`. La mediana sustituye a la nota original solo si respeta la regla de coherencia de `esquema.py` (nota ≤ 2 exige un factor de insatisfacción; nota ≥ 4 no admite 3+ factores); si no, se conserva la original. Se guarda la **dispersión** (máx − mín) para marcar los casos inestables que conviene revisar a mano.

Se **suma** al score híbrido 70/30 y a la prueba de estabilidad de la sección 9, no los reemplaza. La celda de ejecución está comentada: consume `N × K` llamadas.

In [25]:
# ============================================================================
# 7.a AUTOCONSISTENCIA DE satisfaccion_llm  -  la ejecucion consume N x K llamadas
# ============================================================================
CACHE_AUTOCONSISTENCIA = DIR_OUT / "cache_autoconsistencia.jsonl"
N_CANDIDATAS_AUTOCONSISTENCIA = 40      # cola baja del ranking preliminar; basta para fijar las 5 peores
SEMILLAS_AUTOCONSISTENCIA = [202, 7, 1234, 99, 555][:prompts.K_AUTOCONSISTENCIA]


def _muestrear_satisfaccion(cid, transcripcion, semillas):
    """K llamadas extra a TEMP_AUTOCONSISTENCIA; devuelve las notas satisfaccion_llm validas."""
    mensajes = prompts.construir_mensajes(transcripcion, compacto=True, con_few_shot=True)
    notas = []
    for s in semillas:
        crudo, _ = _llamar_api(mensajes, semilla=s, temperatura=prompts.TEMP_AUTOCONSISTENCIA)
        try:
            datos = json.loads(limpiar_respuesta(crudo or ''))
            notas.append(int(ConversacionIA.model_validate(datos).satisfaccion_llm))
        except Exception:
            continue
    return notas


def _coherente(nota, n_factores_reales):
    """La regla de esquema._regla_coherencia_satisfaccion, sobre columnas de conv_ia."""
    if nota <= 2 and n_factores_reales == 0:
        return False
    if nota >= 4 and n_factores_reales >= 3:
        return False
    return True


def muestrear_candidatas(conv_ia, n, ruta_cache):
    """Remuestrea satisfaccion_llm de las n peores del ranking preliminar. Escribe a la cache."""
    hechas = cargar_cache(ruta_cache)
    candidatas = conv_ia.nsmallest(n, "score_satisfaccion").uk_id_conversacion.tolist()
    pendientes = [c for c in candidatas if c not in hechas]
    print(f"Candidatas: {len(candidatas)} | en cache: {len(candidatas) - len(pendientes)} | "
          f"por muestrear: {len(pendientes)}  ({len(pendientes) * prompts.K_AUTOCONSISTENCIA} llamadas)")
    for i, cid in enumerate(pendientes, 1):
        notas = _muestrear_satisfaccion(cid, transcripciones[cid], SEMILLAS_AUTOCONSISTENCIA)
        anexar_cache(ruta_cache, {"uk_id_conversacion": cid, "notas_muestreadas": notas})
        print(f"  [{i:>3}/{len(pendientes)}] {cid[:30]:<30} notas={notas}")
    return cargar_cache(ruta_cache)


def aplicar_autoconsistencia(conv_ia, ruta_cache):
    """Sustituye satisfaccion_llm por la mediana (nota temp0 + K muestras) donde sea coherente."""
    cache = cargar_cache(ruta_cache)
    d = conv_ia.copy()
    if "satisfaccion_llm_original" not in d.columns:
        d["satisfaccion_llm_original"] = d["satisfaccion_llm"]
    d["satisfaccion_dispersion"] = 0
    d["satisfaccion_estabilizada"] = False
    n_remuestreadas = 0
    for idx, fila in d.iterrows():
        reg = cache.get(fila.uk_id_conversacion)
        if not reg or not reg.get("notas_muestreadas"):
            continue
        n_remuestreadas += 1
        notas = [int(fila.satisfaccion_llm_original)] + list(reg["notas_muestreadas"])
        cons = prompts.consolidar_satisfaccion([{"satisfaccion_llm": v} for v in notas])
        d.at[idx, "satisfaccion_dispersion"] = cons["dispersion"]
        mediana = cons["satisfaccion_llm"]
        if mediana != fila.satisfaccion_llm_original and _coherente(mediana, fila.n_factores_insatisfaccion):
            d.at[idx, "satisfaccion_llm"] = mediana
            d.at[idx, "satisfaccion_estabilizada"] = True
    print(f"Remuestreadas           : {n_remuestreadas}")
    print(f"Nota cambiada a mediana : {int(d.satisfaccion_estabilizada.sum())}")
    print(f"Dispersion >= 2         : {int((d.satisfaccion_dispersion >= 2).sum())}  -> revisar a mano")
    return d


# ---- Ejecucion: descomentar. Consume N x K llamadas a la API ---------------
# _ = muestrear_candidatas(conv_ia, N_CANDIDATAS_AUTOCONSISTENCIA, CACHE_AUTOCONSISTENCIA)

if not conv_ia.empty and cargar_cache(CACHE_AUTOCONSISTENCIA):
    conv_ia = aplicar_autoconsistencia(conv_ia, CACHE_AUTOCONSISTENCIA)
    conv_ia['componente_llm'] = (conv_ia.satisfaccion_llm - 1) / 4 * 100
    conv_ia['score_satisfaccion'] = (PESO_LLM * conv_ia.componente_llm
                                     + PESO_OBJETIVO * conv_ia.componente_objetivo).round(1)
    print("\n(e) Las 5 conversaciones peor calificadas (con autoconsistencia aplicada):")
    display(conv_ia.nsmallest(5, "score_satisfaccion")[
        ["uk_id_conversacion", "score_satisfaccion", "satisfaccion_llm",
         "satisfaccion_llm_original", "satisfaccion_dispersion",
         "senales_activas", "factores_insatisfaccion", "recomendacion_mejora"]])
else:
    print("Autoconsistencia lista. Descomente muestrear_candidatas(...) para ejecutarla "
          f"(~{N_CANDIDATAS_AUTOCONSISTENCIA * prompts.K_AUTOCONSISTENCIA} llamadas).")
    print("Sin cache: el ranking (e) definitivo es el preliminar de la seccion 7.")

Autoconsistencia lista. Descomente muestrear_candidatas(...) para ejecutarla (~120 llamadas).
Sin cache: el ranking (e) definitivo es el preliminar de la seccion 7.


### 7.b Validación externa con el NPS que recoge el propio bot

Hay una fuente de verdad independiente escondida en los datos. En una parte de las conversaciones el bot cierra preguntando:

> *"EN UNA ESCALA DEL 0 AL 10, ¿QUÉ TAN PROBABLE ES QUE RECOMIENDES NUESTRO SERVICIO?"*

Cuando el cliente responde con un número, eso es **satisfacción declarada por el propio cliente**, no inferida por un modelo. Es la mejor validación externa disponible para el score, y no cuesta ninguna llamada a la API.

**Precaución metodológica necesaria:** el corpus también contiene menús del bot con opciones numeradas (*"1 OPCIÓN: AUTORIZACIÓN DE DATOS"*), así que un mensaje del usuario que sea sólo un dígito **no es necesariamente un NPS**. Por eso sólo se acepta la respuesta numérica cuando aparece **inmediatamente después** de la pregunta de NPS y está en el rango 0-10. Sin ese filtro, la validación se contamina con selecciones de menú.

Esta validación es **complementaria y no alimenta el score** — mantenerla fuera es lo que le permite servir de verificación independiente. Además sólo cubre una fracción del corpus, así que usarla como componente haría el score inconsistente entre conversaciones con y sin NPS.

*Nota de la corrida final: al recalcular esta comparación sobre los datos de `gpt-4o-mini` (`docs/BRIEFING_RAG.md`), el LLM solo quedó más cerca del NPS declarado que el score híbrido — ver el comentario en la sección 7.*

In [26]:
PATRON_NPS = r"ESCALA DEL? 0 AL 10|ESCALA DE 0 AL 10"


def extraer_nps(limpio: pd.DataFrame) -> pd.DataFrame:
    """NPS declarado: digito 0-10 del USUARIO inmediatamente despues de la pregunta."""
    filas = []
    for cid, sub in limpio.groupby("uk_id_conversacion", sort=False):
        sub = sub.sort_values("orden").reset_index(drop=True)
        pregunta = sub.msg.str.contains(PATRON_NPS, regex=True, na=False)
        for i in sub.index[pregunta]:
            for j in range(i + 1, min(i + 4, len(sub))):
                if sub.at[j, "de"] != "USUARIO":
                    continue
                texto = str(sub.at[j, "msg"]).strip()
                if re.fullmatch(r"\d{1,2}", texto) and 0 <= int(texto) <= 10:
                    filas.append({"uk_id_conversacion": cid, "nps": int(texto),
                                  "distancia": j - i})
                    break
            else:
                continue
            break
    return pd.DataFrame(filas)


nps = extraer_nps(limpio)
convs_con_pregunta = limpio[limpio.msg.str.contains(PATRON_NPS, regex=True, na=False)].uk_id_conversacion.nunique()

print(f"Conversaciones donde el bot pregunta NPS : {convs_con_pregunta:>5} de {limpio.uk_id_conversacion.nunique():,}")
print(f"Con respuesta numerica valida del cliente: {len(nps):>5} ({len(nps)/limpio.uk_id_conversacion.nunique()*100:.1f}% del corpus)")
if len(nps):
    print()
    print("Distribucion del NPS declarado:")
    print(nps.nps.value_counts().sort_index().to_string())
    detr = (nps.nps <= 6).mean() * 100
    prom = (nps.nps >= 9).mean() * 100
    print(f"\n  Detractores (0-6): {detr:.1f}%   Promotores (9-10): {prom:.1f}%   NPS neto: {prom-detr:+.1f}")

Conversaciones donde el bot pregunta NPS :   746 de 1,197
Con respuesta numerica valida del cliente:   236 (19.7% del corpus)

Distribucion del NPS declarado:
nps
0      40
1       3
3       3
4       1
5       4
6       3
7       1
8      12
9      17
10    152

  Detractores (0-6): 22.9%   Promotores (9-10): 71.6%   NPS neto: +48.7


In [27]:
# Correlacion del score con el NPS. Es la prueba de que el score mide algo real
# y no solo su propia definicion.
if not conv_ia.empty and len(nps):
    val = conv_ia.merge(nps, on="uk_id_conversacion", how="inner")
    print(f"Conversaciones con score y NPS simultaneamente: {len(val)}")
    if len(val) >= 10:
        from scipy.stats import spearmanr
        for col, etiqueta in [("score_satisfaccion", "score hibrido"),
                              ("componente_llm", "solo LLM"),
                              ("componente_objetivo", "solo senales objetivas")]:
            rho, p = spearmanr(val[col], val.nps)
            print(f"  {etiqueta:<24} rho de Spearman = {rho:+.3f}   p = {p:.4f}")
        print()
        print("Lectura: si el score hibrido correlaciona mejor con el NPS declarado")
        print("que sus dos componentes por separado, la combinacion 70/30 esta")
        print("aportando. Si no, hay que revisar los pesos.")
        display(val.groupby(pd.cut(val.nps, [-1, 6, 8, 10], labels=["detractor", "pasivo", "promotor"]),
                            observed=True).score_satisfaccion.agg(["count", "mean", "std"]).round(1))
    else:
        print("Muestra insuficiente para correlacionar. Se requiere la corrida completa.")
else:
    print("Requiere extraccion ejecutada. Sin datos, no se reporta ninguna correlacion.")

Conversaciones con score y NPS simultaneamente: 4
Muestra insuficiente para correlacionar. Se requiere la corrida completa.


---
## 8. Validación contra anotación humana

Un LLM que se autoevalúa no demuestra nada. La única evidencia real de que esta capa funciona es comparar sus salidas contra un criterio humano sobre una muestra que el modelo no eligió.

### Diseño

**Muestra estratificada de 60 conversaciones**, no aleatoria simple. Los estratos cruzan `acuerdo_pago` (2 niveles) con tramo de satisfacción (3 niveles), 10 por celda. El motivo: los acuerdos de pago son minoritarios, y una muestra aleatoria simple traería tan pocos casos positivos que el kappa de `acuerdo_pago` —justo la variable más crítica— se estimaría con un intervalo de confianza inservible.

### Métrica

**Kappa de Cohen**, no accuracy. En una variable desbalanceada, un clasificador que responda siempre "no hay acuerdo" puede alcanzar 85% de accuracy sin haber aprendido nada. El kappa descuenta el acuerdo esperable por azar. Se reporta también la accuracy, pero la decisión se toma con el kappa.

### Criterios de aceptación, fijados antes de medir

| Variable | Umbral | Justificación |
|---|---|---|
| `acuerdo_pago` | **κ ≥ 0,75** | Es el denominador de las preguntas (b) y (c). Un error aquí se propaga a todo el análisis de efectividad. Exige acuerdo *sustancial a casi perfecto* |
| `motivo_no_pago` | **κ ≥ 0,60** | Trece categorías, con fronteras genuinamente difusas —"desempleo" contra "ingresos insuficientes" es discutible incluso entre dos anotadores humanos. Se acepta acuerdo *moderado a sustancial* |

Escala de referencia de Landis y Koch: <0,20 pobre · 0,21-0,40 débil · 0,41-0,60 moderado · 0,61-0,80 sustancial · >0,80 casi perfecto.

**Si un umbral no se alcanza, la acción no es rebajar el umbral**: es revisar los desacuerdos, identificar el patrón, corregir el prompt o la taxonomía, y volver a medir. Esa iteración pertenece a la bitácora de `prompts.py`.

In [28]:
RUTA_MUESTRA = DIR_OUT / "muestra_anotacion.csv"
RUTA_ANOTADA = DIR_OUT / "muestra_anotacion_COMPLETADA.csv"


def muestra_estratificada(conv_ia: pd.DataFrame, n_total: int = 60, semilla: int = 42) -> pd.DataFrame:
    """Estratifica por acuerdo_pago x tramo de satisfaccion.

    Si un estrato tiene menos casos de los que le tocan, el remanente se
    redistribuye entre los demas para no devolver una muestra corta.
    """
    d = conv_ia.copy()
    d["tramo_satisfaccion"] = pd.cut(d.satisfaccion_llm, [0, 2, 3, 5],
                                     labels=["baja_1_2", "media_3", "alta_4_5"])
    d["estrato"] = d.acuerdo_pago.astype(str) + " | " + d.tramo_satisfaccion.astype(str)

    grupos = list(d.groupby("estrato", observed=True))
    cupo = n_total // max(len(grupos), 1)
    partes, deficit = [], 0
    for _, g in grupos:
        n = min(cupo, len(g))
        deficit += cupo - n
        partes.append(g.sample(n, random_state=semilla))
    muestra = pd.concat(partes) if partes else d.head(0)

    if deficit > 0:
        resto = d[~d.uk_id_conversacion.isin(muestra.uk_id_conversacion)]
        if len(resto):
            muestra = pd.concat([muestra, resto.sample(min(deficit, len(resto)), random_state=semilla)])
    return muestra


if not conv_ia.empty:
    muestra = muestra_estratificada(conv_ia, n_total=60)
    print(f"Muestra: {len(muestra)} conversaciones")
    display(muestra.groupby(["acuerdo_pago", "tramo_satisfaccion"], observed=True)
                   .size().to_frame("n"))

    # El archivo lleva la transcripcion completa para que el anotador no tenga
    # que ir a buscarla, y las columnas del modelo van al final para reducir el
    # anclaje: se anota primero, se compara despues.
    export = muestra[["uk_id_conversacion"]].copy()
    export["transcripcion"] = export.uk_id_conversacion.map(transcripciones)
    export["h_acuerdo_pago"] = ""          # anotar: True / False
    export["h_motivo_no_pago"] = ""        # anotar: una categoria de MotivoNoPago
    export["h_satisfaccion"] = ""          # anotar: 1 a 5
    export["h_observaciones"] = ""
    for c in ["acuerdo_pago", "motivo_no_pago", "satisfaccion_llm",
              "fecha_compromiso_texto", "evidencia_motivo"]:
        export[f"ia_{c}"] = muestra[c].values

    export.to_csv(RUTA_MUESTRA, index=False, encoding="utf-8-sig")
    print(f"\nGuardado: outputs/{RUTA_MUESTRA.name}")
    print("Instrucciones: llene las columnas h_* leyendo la transcripcion, SIN mirar")
    print("las columnas ia_*, y guarde como", RUTA_ANOTADA.name)
    print("Categorias validas de motivo:", ", ".join(m.value for m in esquema.MotivoNoPago))
else:
    print("Requiere extraccion ejecutada para construir la muestra.")

Muestra: 15 conversaciones


n
acuerdo_pago tramo_satisfaccion   
False        baja_1_2            6
             media_3             1
             alta_4_5            4
True         media_3             1
             alta_4_5            3


Guardado: outputs/muestra_anotacion.csv
Instrucciones: llene las columnas h_* leyendo la transcripcion, SIN mirar
las columnas ia_*, y guarde como muestra_anotacion_COMPLETADA.csv
Categorias validas de motivo: YA_PAGO_O_PAGO_NO_APLICADO, DESCUENTO_NOMINA_NO_APLICADO, DESEMPLEO, INGRESOS_INSUFICIENTES, SOBREENDEUDAMIENTO, ENFERMEDAD_O_CALAMIDAD, NEGOCIO_O_INGRESO_VARIABLE, DESACUERDO_MONTO_O_COBRO, SINIESTRO_SEGURO_EN_TRAMITE, NO_RECONOCE_LA_OBLIGACION, OLVIDO_O_DESCONOCIMIENTO, OTRO, NINGUNO


In [29]:
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix

CRITERIOS = {"acuerdo_pago": 0.75, "motivo_no_pago": 0.60}


def evaluar_contra_anotacion(ruta: Path) -> pd.DataFrame | None:
    """Calcula accuracy y kappa contra las anotaciones humanas.

    Devuelve None si el archivo anotado aun no existe. NO inventa metricas.
    """
    if not Path(ruta).exists():
        print(f"No existe {ruta.name}.")
        print("La validacion contra anotacion humana NO se ha realizado todavia.")
        print("Ninguna afirmacion sobre la precision de esta capa esta respaldada")
        print("mientras este archivo no exista y esta celda no se ejecute.")
        return None

    a = pd.read_csv(ruta)
    filas = []
    for var, umbral in CRITERIOS.items():
        col_h, col_ia = f"h_{var if var != 'motivo_no_pago' else 'motivo_no_pago'}", f"ia_{var}"
        if col_h not in a.columns or col_ia not in a.columns:
            print(f"Faltan columnas para {var}: {col_h} / {col_ia}")
            continue
        sub = a[[col_h, col_ia]].dropna()
        sub = sub[sub[col_h].astype(str).str.strip() != ""]
        if len(sub) < 10:
            print(f"{var}: solo {len(sub)} anotaciones, insuficiente para estimar kappa.")
            continue
        y_h = sub[col_h].astype(str).str.strip().str.upper()
        y_ia = sub[col_ia].astype(str).str.strip().str.upper()
        k = cohen_kappa_score(y_h, y_ia)
        filas.append({
            "variable": var, "n": len(sub),
            "accuracy": round(accuracy_score(y_h, y_ia), 3),
            "kappa": round(k, 3), "umbral": umbral,
            "cumple": "SI" if k >= umbral else "NO",
        })

    if "h_satisfaccion" in a.columns:
        sub = a[["h_satisfaccion", "ia_satisfaccion_llm"]].dropna()
        if len(sub) >= 10:
            y_h = sub.h_satisfaccion.astype(float).round().astype(int)
            y_ia = sub.ia_satisfaccion_llm.astype(int)
            filas.append({
                "variable": "satisfaccion (kappa ponderado)", "n": len(sub),
                "accuracy": round((y_h == y_ia).mean(), 3),
                "kappa": round(cohen_kappa_score(y_h, y_ia, weights="quadratic"), 3),
                "umbral": np.nan, "cumple": "-",
            })
            print(f"Satisfaccion: {(abs(y_h - y_ia) <= 1).mean()*100:.0f}% de los casos "
                  f"dentro de +/-1 nivel del anotador humano.")
    return pd.DataFrame(filas) if filas else None


resultado_kappa = evaluar_contra_anotacion(RUTA_ANOTADA)
if resultado_kappa is not None:
    display(resultado_kappa)

No existe muestra_anotacion_COMPLETADA.csv.
La validacion contra anotacion humana NO se ha realizado todavia.
Ninguna afirmacion sobre la precision de esta capa esta respaldada
mientras este archivo no exista y esta celda no se ejecute.


In [30]:
# Verificacion de que la celda de kappa funciona, con vectores de juguete.
# No dice nada sobre la calidad de la extraccion: solo prueba la aritmetica.
_h = ["True"]*20 + ["False"]*30
_ia = ["True"]*18 + ["False"]*2 + ["False"]*28 + ["True"]*2
print("Prueba de la metrica con datos sinteticos (NO son resultados del modelo):")
print(f"  accuracy = {accuracy_score(_h, _ia):.3f}")
print(f"  kappa    = {cohen_kappa_score(_h, _ia):.3f}")
print(f"  criterio acuerdo_pago (>= {CRITERIOS['acuerdo_pago']}): "
      f"{'cumpliria' if cohen_kappa_score(_h,_ia) >= CRITERIOS['acuerdo_pago'] else 'no cumpliria'}")
print("\nMatriz de confusion:")
print(pd.DataFrame(confusion_matrix(_h, _ia), index=["real:False","real:True"],
                   columns=["pred:False","pred:True"]).to_string())

Prueba de la metrica con datos sinteticos (NO son resultados del modelo):
  accuracy = 0.920
  kappa    = 0.833
  criterio acuerdo_pago (>= 0.75): cumpliria

Matriz de confusion:
            pred:False  pred:True
real:False          28          2
real:True            2         18


---
## 9. Prueba de estabilidad

`temperature=0` **no garantiza determinismo**. La inferencia por lotes en GPU introduce no-determinismo en la reducción de punto flotante, así que la misma petición puede dar salidas ligeramente distintas. Con `seed` fijo la variación se reduce, pero no desaparece.

Esto importa por una razón práctica: si el mismo texto produce categorías distintas entre corridas, los conteos de las preguntas (a), (b) y (c) no son reproducibles y cualquier conclusión que se saque de ellos es frágil.

**Diseño:** 10 conversaciones, 3 corridas cada una, con semillas distintas para medir el peor caso. Se reporta el porcentaje de coincidencia campo por campo.

Lectura de los resultados:
- Los campos categóricos y booleanos deberían superar el **90%**. Por debajo de eso, la categoría tiene fronteras mal definidas y hay que revisar su descripción en el prompt.
- `resumen`, `justificacion_satisfaccion` y `recomendacion_mejora` son texto libre: **no se espera que coincidan literalmente** y se excluyen de la métrica. Su estabilidad tendría que medirse por similitud semántica, que es otro ejercicio.

In [31]:
CAMPOS_ESTABILIDAD = [
    "motivo_no_pago", "acuerdo_pago", "tipo_fecha", "satisfaccion_llm",
    "cliente_reconoce_deuda", "asesor_pregunta_por_motivo",
    "sentimiento_inicial", "sentimiento_final",
    "hubo_friccion_bot", "conversacion_abandonada",
]
IDS_ESTABILIDAD = list(tr.sample(10, random_state=7).uk_id_conversacion)
SEMILLAS = [42, 123, 2024]

print("Conversaciones de la prueba:", len(IDS_ESTABILIDAD))
print("Corridas por conversacion  :", len(SEMILLAS))
print(f"Llamadas que consumiria    : {len(IDS_ESTABILIDAD)*len(SEMILLAS)}")

Conversaciones de la prueba: 10
Corridas por conversacion  : 3
Llamadas que consumiria    : 30


In [32]:
# EJECUTABLE. Consume cuota: 30 llamadas.
# for s in SEMILLAS:
#     ruta = DIR_OUT / f"cache_estabilidad_s{s}.jsonl"
#     procesar(IDS_ESTABILIDAD, ruta, semilla=s, etiqueta=f"Estabilidad semilla {s}: ")
print("Celda de estabilidad lista. Descomente para ejecutarla (30 llamadas).")

Celda de estabilidad lista. Descomente para ejecutarla (30 llamadas).


In [33]:
def medir_estabilidad(ids, semillas, campos):
    """% de conversaciones donde las N corridas coinciden, campo por campo."""
    corridas = []
    for s in semillas:
        c = cargar_cache(DIR_OUT / f"cache_estabilidad_s{s}.jsonl")
        if c:
            corridas.append(c)
    if len(corridas) < 2:
        print("Se necesitan al menos 2 corridas en cache. Ejecute la celda anterior.")
        return None

    filas = []
    for campo in campos:
        coincidencias, evaluadas = 0, 0
        for cid in ids:
            valores = [c[cid]["datos"][campo] for c in corridas
                       if cid in c and c[cid].get("ok") and c[cid].get("datos")]
            if len(valores) == len(corridas):
                evaluadas += 1
                coincidencias += len(set(map(str, valores))) == 1
        if evaluadas:
            filas.append({"campo": campo, "n_evaluadas": evaluadas,
                          "coincidencia_%": round(coincidencias / evaluadas * 100, 1)})
    res = pd.DataFrame(filas).sort_values("coincidencia_%")
    print(f"Corridas comparadas: {len(corridas)}")
    print(f"Coincidencia media sobre campos estructurados: {res['coincidencia_%'].mean():.1f}%")
    print("\nCampos por debajo del 90% -> revisar su definicion en el prompt:")
    bajos = res[res["coincidencia_%"] < 90]
    print("  ninguno" if bajos.empty else bajos.to_string(index=False))
    return res


estabilidad = medir_estabilidad(IDS_ESTABILIDAD, SEMILLAS, CAMPOS_ESTABILIDAD)
if estabilidad is not None:
    display(estabilidad)

Se necesitan al menos 2 corridas en cache. Ejecute la celda anterior.


---
## 10. Exportación

Cinco archivos en `outputs/`. Las tres tablas largas están en el formato que necesitan tanto el análisis de efectividad como el modelo relacional de Power BI: `uk_id_conversacion` es la llave que relaciona la tabla de hechos (`conversaciones_ia`) con las de detalle en cardinalidad 1:N.

| Archivo | Grano | Para qué |
|---|---|---|
| `conversaciones_ia.csv` | 1 fila por conversación | Tabla de hechos. Preguntas (a), (d), (e) |
| `ofertas_largo.csv` | 1 fila por conversación-oferta | Preguntas (b) y (c) |
| `argumentos_largo.csv` | 1 fila por conversación-argumento | Pregunta (c) |
| `factores_largo.csv` | 1 fila por conversación-factor | Pregunta (e) |
| `transcripciones.csv` | 1 fila por conversación | Auditoría: permite verificar cualquier variable contra su texto de origen |

Se exporta con `utf-8-sig` porque Excel y Power BI en Windows interpretan mal el UTF-8 sin BOM y rompen los acentos.

In [34]:
def exportar():
    if conv_ia.empty:
        print("No hay datos extraidos. No se exporta nada para no dejar archivos vacios")
        print("que despues se confundan con resultados reales.")
        print("\nSi ya existe transcripciones.csv, ese si esta completo (no depende del LLM).")
        return

    salidas = {
        "conversaciones_ia.csv": conv_ia,
        "ofertas_largo.csv": ofertas_largo,
        "argumentos_largo.csv": argumentos_largo,
        "factores_largo.csv": factores_largo,
        "transcripciones.csv": tr,
    }
    print("Exportado a outputs/:")
    for nombre, tabla in salidas.items():
        tabla.to_csv(DIR_OUT / nombre, index=False, encoding="utf-8-sig")
        print(f"  {nombre:<26} {len(tabla):>6,} filas x {len(tabla.columns):>2} columnas")

    cobertura = len(conv_ia) / len(tr) * 100
    print(f"\nCobertura de la extraccion: {len(conv_ia):,} de {len(tr):,} conversaciones ({cobertura:.1f}%)")
    if cobertura < 100:
        print("ATENCION: cobertura parcial. Cualquier conteo agregado se refiere solo")
        print("a las conversaciones efectivamente procesadas, no al corpus completo.")


exportar()

Exportado a outputs/:
  conversaciones_ia.csv          15 filas x 44 columnas
  ofertas_largo.csv              10 filas x  4 columnas
  argumentos_largo.csv           13 filas x  4 columnas
  factores_largo.csv             23 filas x  3 columnas
  transcripciones.csv         1,197 filas x  5 columnas

Cobertura de la extraccion: 15 de 1,197 conversaciones (1.3%)
ATENCION: cobertura parcial. Cualquier conteo agregado se refiere solo
a las conversaciones efectivamente procesadas, no al corpus completo.


---
## Estado de esta capa y qué falta

La extracción ya corrió sobre el corpus completo: **1.197 de 1.197
conversaciones, sin un solo fallo de validación**, con `gpt-4o-mini` y el
contrato de `esquema.py` con el validador de rol del acuerdo incluido
(bitácora v13 en `prompts.py`). `conversaciones_ia.csv` quedó con 48
columnas — las variables de negocio más las de trazabilidad que dejó cada
corrección (qué dijo el modelo antes de aplicar la regla, y si se
reclasificó).

Para que quede constancia de por qué importó cambiar de modelo:

| | `gpt-4.1-nano` (1ª corrida) | `gpt-4o-mini` (corrida final) |
|---|---|---|
| `motivo_no_pago` = NINGUNO | 94% | 63% |
| `acuerdo_pago` confirmado por el cliente | 5,4% | 6,5% |

El salto en `motivo_no_pago` es la diferencia entre una pregunta (a) casi sin
respuesta y una con una distribución que se puede presentar y discutir.

**Lo que todavía falta, sin adornos:** nada de esto está medido contra un
criterio humano. Los umbrales de kappa de la sección 8 (κ ≥ 0,75 para
`acuerdo_pago`, κ ≥ 0,60 para `motivo_no_pago`) siguen siendo criterios
fijados de antemano, no resultados. Falta anotar a mano la muestra de 60,
calcular el kappa, y decidir desde ahí si el modelo actual alcanza o si hace
falta seguir ajustando el prompt, la taxonomía, o correr la autoconsistencia
de la sección 7.a para estabilizar el ranking de la pregunta (e) — el ranking
de las 5 peor calificadas cambió bastante entre `nano` y `gpt-4o-mini`, señal
de que una sola corrida no alcanza para fijarlo con confianza.

**Limitaciones que ninguna corrida resuelve:**

1. El corpus está truncado por diseño — mínimo 20 mensajes por conversación.
   Las interacciones cortas, que en una operación real son la mayoría, no
   están representadas.
2. Las tasas de conversión por oferta son observacionales: el asesor elige
   qué ofrecer según el caso, así que el tipo de oferta está confundido con
   la disposición previa del cliente. Que una oferta muestre una tasa alta de
   acuerdo puede significar que funciona o que solo se ofrece a quien ya iba
   a pagar. Separar ambas cosas exige un experimento, no más análisis del
   histórico.
3. Los pesos del score híbrido son una hipótesis fundamentada, no una
   calibración — y contra el NPS declarado corre peor que el juicio del LLM
   solo (sección 7.b). Vale la pena reportar los dos números por separado.
4. Sin eje temporal: `anio` y `mes` son constantes en toda la base.